In [ ]:
# MATLAB section 1/43 for nSTATPaperExamples: nSTAT J. Neuroscience Methods Paper Examples

# % nSTAT J. Neuroscience Methods Paper Examples
#
# *Author*: Iahn Cajigas
#
# *Date*: 01/04/2012
#
# Force command echo off so published output does not include repeated
# executed source lines.
# MATLAB: echo off;
#
# Paper reference:
#
# * Cajigas I, Malik WQ, Brown EN. nSTAT: Open-source neural spike train
# analysis toolbox for Matlab. Journal of Neuroscience Methods 211:
# 245-264 (2012).
# * DOI: 10.1016/j.jneumeth.2012.08.009
# * PMID: 22981419
#
# Navigation:
#
# * <PaperOverview.html Paper-Aligned Toolbox Map>
#
#

# Python translation bootstrap for this helpfile.

# Topic: nSTATPaperExamples
# Execution group: smoke
# Workflow family: decoding_1d
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/nSTATPaperExamples.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "nSTATPaperExamples"
FAMILY = "decoding_1d"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"nSTATPaperExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"nSTATPaperExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"nSTATPaperExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"nSTATPaperExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB section 2/43 for nSTATPaperExamples: Experiment 1

# % Experiment 1
# MINIATURE EXCITATORY POST-SYNAPTIC CURRENTS (mEPSCs)
# Data from Marnie Phillips  <marnie.a.phillips@gmail.com>
# This analysis is based on a partial version of the dataset used in
#
# Phillips MA, Lewis LD, Gong J, Constantine-Paton M, Brown EN.  2011
# _Model-based statistical analysis of miniature synaptic transmission._
# J Neurophys (under consideration)
#
# *Date*: 03/01/2011

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/43 for nSTATPaperExamples: Constant Magnesium Concentration - Constant rate poisson

# % Constant Magnesium Concentration - Constant rate poisson
# Under a constant Magnesium concentration, it is seen that the mEPSCs
# behave as a homogeneous poisson process (constant arrival rate).
# MATLAB:     close all; clear all;
# MATLAB:     [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:         getPaperDataDirs();
# MATLAB:     nSTATRootDir = fileparts(dataDir);
# MATLAB:     if exist(nSTATRootDir,'dir') == 7 && ~strcmp(pwd,nSTATRootDir)
# MATLAB:         cd(nSTATRootDir);
# MATLAB:     end
# MATLAB:     epsc2 = importdata(fullfile(mEPSCDir,'epsc2.txt'));
# MATLAB:     sampleRate = 1000;
# MATLAB:     spikeTimes = epsc2.data(:,2)*1/sampleRate; %in seconds
# MATLAB:     nstConst = nspikeTrain(spikeTimes);
# MATLAB:     time = 0:(1/sampleRate):nstConst.maxTime;
#
#
# Define Covariates for the analysis
# MATLAB:     baseline = Covariate(time,ones(length(time),1),'Baseline','time','s',...
# MATLAB:         '',{'\mu'});
# MATLAB:     covarColl = CovColl({baseline});
#
# Create the trial structure
# MATLAB:     spikeColl = nstColl(nstConst);
# MATLAB:     trial     = Trial(spikeColl,covarColl);
#
#
# Define how we want to analyze the data
# MATLAB:     clear tc tcc;
# MATLAB:     tc{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,[]);
# MATLAB:     tc{1}.setName('Constant Baseline');
# MATLAB:     tcc = ConfigColl(tc);
#
# Perform Analysis (Commented to since data already saved)
# MATLAB:     results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
# h=results.plotResults;
# MATLAB:     close all;
# MATLAB:     scrsz = get(0,'ScreenSize');
# MATLAB:     results.lambda.setDataLabels({'\lambda_{const}'});
# MATLAB:     h=figure('OuterPosition',[scrsz(3)*.01 scrsz(4)*.04 ...
# MATLAB:         scrsz(3)*.98 scrsz(4)*.95]);
#
# MATLAB:     subplot(2,2,1); spikeColl.plot;
# MATLAB:         title({'Neural Raster with constant Mg^{2+} Concentration'},...
# MATLAB:             'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB:          hx=xlabel('time [s]','Interpreter','none');
# MATLAB:          hy=ylabel('mEPSCs','Interpreter','none');
# MATLAB:          set(gca,'yTick',[0 1]);
# MATLAB:          set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:     subplot(2,2,3); results.KSPlot;
# MATLAB:     subplot(2,2,2); results.plotInvGausTrans;
# MATLAB:     subplot(2,2,4); results.lambda.plot([],{{' ''b'' ,''Linewidth'',2'}});
# MATLAB:     hx=xlabel('time [s]','Interpreter','none');
# MATLAB:     hy=get(gca,'YLabel');
# MATLAB:     set([hx hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:     h_legend = legend('\lambda_{const}','Location','NorthEast');
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)+.05 pos(2) pos(3:4)]);
# MATLAB:     set(h_legend,'FontSize',14)
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/43 for nSTATPaperExamples: Varying Magnesium Concentration - Piecewise Constant rate poisson

# % Varying Magnesium Concentration - Piecewise Constant rate poisson
# When the magnesium concentration of the bath decreased (i.e. magnesium
# is removed), the rate of mEPSCs begin to increase in frequency. This can
# be modeled in a many different ways (using the change in Magnesium
# directly as a model covariate, etc.) Here we approximate the rate as
# being constant during certain portions of the experiment. These segments
# can in principle be estimated (using heirarchical Bayesian methods), but
# here we select them via visual inspection. We compare three models: a
# constant rate model (from above), a piecewise constant rate model, and a
# piecewise constant rate model with history.
# MATLAB:     close all;
# load the data;
# MATLAB:     washout1 = importdata(fullfile(mEPSCDir,'washout1.txt'));
# MATLAB:     washout2 = importdata(fullfile(mEPSCDir,'washout2.txt'));
#
# MATLAB:     sampleRate  = 1000;
# Magnesium removed at t=0
# MATLAB:     spikeTimes1 = 260+washout1.data(:,2)*1/sampleRate; %in seconds
# MATLAB:     spikeTimes2 = sort(washout2.data(:,2))*1/sampleRate + 745;%in seconds
# MATLAB:     nst = nspikeTrain([spikeTimes1; spikeTimes2]);
# MATLAB:     time = 260:(1/sampleRate):nst.maxTime;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/43 for nSTATPaperExamples: Data Visualization

# % Data Visualization
# Visual inspection of the spike train is used to pick three regions
# where the firing rate appears to be different. Here we do not
# estimate where these transitions happen but pick times in an ad-hoc
# manner.
# MATLAB:     scrsz = get(0,'ScreenSize');
# MATLAB:     h=figure('OuterPosition',[scrsz(3)*.01 scrsz(4)*.04 scrsz(3)*.6 ...
# MATLAB:         scrsz(4)*.9]);
#
# MATLAB:     subplot(2,1,1);
# MATLAB:     nstConst.plot; set(gca,'yTick',[0 1]); hy=ylabel('mEPSCs');
# MATLAB:      title({'Neural Raster with constant Mg^{2+} Concentration'},...
# MATLAB:          'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB:     hx=get(gca,'XLabel');
# MATLAB:     set([hx,hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# MATLAB:     subplot(2,1,2);
# MATLAB:     nst.plot; set(gca,'yTick',[0 1]); hy=ylabel('mEPSCs');
# MATLAB:     title({'Neural Raster with decreasing Mg^{2+} Concentration'},...
# MATLAB:         'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB:     hx=get(gca,'XLabel');
# MATLAB:     set([hx,hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/43 for nSTATPaperExamples: Define Covariates for the analysis

# % Define Covariates for the analysis
# MATLAB:           timeInd1 =find(time<495,1,'last'); %0-495sec first constant rate
# MATLAB:         timeInd2 =find(time<765,1,'last'); %495-765 second constant rate epoch
# 765 onwards third constant rate
# epoch
# MATLAB:     constantRate = ones(length(time),1);
# MATLAB:     rate1 = zeros(length(time),1); rate1(1:timeInd1)=1;
# MATLAB:     rate2 = zeros(length(time),1); rate2((timeInd1+1):timeInd2)=1;
# MATLAB:     rate3 = zeros(length(time),1); rate3((timeInd2+1):end)=1;
# MATLAB:     baseline = Covariate(time,[constantRate,rate1, rate2, rate3],...
# MATLAB:         'Baseline','time','s','',{'\mu','\mu_{1}','\mu_{2}','\mu_{3}'});
# MATLAB:     covarColl = CovColl({baseline});
#
# Create the trial structure
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     trial     = Trial(spikeColl,covarColl);
#
# 30ms history in logarithmic spacing (chosen after using
# Analysis.computeHistLagForAll for various window lengths)
# MATLAB:     maxWindow=.3; numWindows=20;
# MATLAB:     delta=1/sampleRate;
# MATLAB:     windowTimes =unique(round([0 logspace(log10(delta),...
# MATLAB:     log10(maxWindow),numWindows)]*sampleRate)./sampleRate);
# MATLAB:     windowTimes = windowTimes(1:11);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/43 for nSTATPaperExamples: Define how we want to analyze the data

# % Define how we want to analyze the data
# MATLAB:     clear tc tcc;
# MATLAB:     tc{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,[]);
# MATLAB:     tc{1}.setName('Constant Baseline');
# MATLAB:     tc{2} = TrialConfig({{'Baseline','\mu_{1}','\mu_{2}','\mu_{3}'}},...
# MATLAB:         sampleRate,[]); tc{2}.setName('Diff Baseline');
# MATLAB:     tcc = ConfigColl(tc);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 7

_ = section_index


In [ ]:
# MATLAB section 8/43 for nSTATPaperExamples: Perform Analysis

# % Perform Analysis
# We see that the piece-wise constant rate model (without
# history) outperforms the constant baseline model in terms of AIC, BIC,
# and KS-statistic.
# MATLAB:     results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
# h=results.plotResults;
# Summary = FitResSummary(results);
# h=Summary.plotSummary;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 8

_ = section_index


In [ ]:
# MATLAB section 9/43 for nSTATPaperExamples: Section

# %
# MATLAB: close all;
# MATLAB:     scrsz = get(0,'ScreenSize');
# MATLAB:     results.lambda.setDataLabels({'\lambda_{const}',...
# MATLAB:         '\lambda_{const-epoch}'});
# MATLAB:     h=figure('OuterPosition',[scrsz(3)*.01 scrsz(4)*.04 ...
# MATLAB:         scrsz(3)*.98 scrsz(4)*.95]);
#
# MATLAB:     subplot(2,2,1); spikeColl.plot;
# MATLAB:         title({'Neural Raster with decreasing Mg^{2+} Concentration'},...
# MATLAB:             'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB:         hx=xlabel('time [s]','Interpreter','none');
# MATLAB:         set(gca,'YTickLabel',[]);
# MATLAB:         set([hx],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:         timeInd1 =find(time<495,1,'last'); %0-495sec first constant rate
# MATLAB:         timeInd2 =find(time<765,1,'last'); %495-765 second constant rate epoch
# 765 onwards third constant rate
# epoch
# MATLAB:         plot([495;495],[0,1],'r','Linewidth',4); hold on;
# MATLAB:         plot([765;765],[0,1],'r','Linewidth',4);
#
# MATLAB:     subplot(2,2,3); results.KSPlot;
# MATLAB:     subplot(2,2,2); results.plotInvGausTrans;
# MATLAB:     subplot(2,2,4);
# MATLAB:     results.lambda.getSubSignal(1).plot([],{{' ''b'' ,''Linewidth'',2'}});
# MATLAB:     results.lambda.getSubSignal(2).plot([],{{' ''g'' ,''Linewidth'',2'}});
# MATLAB:                     v=axis; axis([v(1) v(2) 0 5]);
# MATLAB:     hx=xlabel('time [s]','Interpreter','none');
# MATLAB:     hy=get(gca,'YLabel');
# MATLAB:     set([hx hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:     h_legend = legend('\lambda_{const}','\lambda_{const-epoch}',...
# MATLAB:         'Location','NorthEast');
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)+.05 pos(2)-.01 pos(3:4)]);
# MATLAB:     set(h_legend,'FontSize',14)
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 9

_ = section_index


In [ ]:
# MATLAB section 10/43 for nSTATPaperExamples: Experiment 2

# % Experiment 2
# EXPLICIT STIMULUS EXAMPLE - WHISKER STIMULATION/THALAMIC NEURON
# In the worksheet with analyze the stimulus effect and history effect on
# the firing of a thalamic neuron under a known stimulus consisting of
# whisker stimulation.
# Data from Demba Ba (demba@mit.edu)
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 10

_ = section_index


In [ ]:
# MATLAB section 11/43 for nSTATPaperExamples: Load the data

# % Load the data
# clear all;
# MATLAB: close all;
# MATLAB: [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:     getPaperDataDirs();
# MATLAB: nSTATRootDir = fileparts(dataDir);
# MATLAB: if exist(nSTATRootDir,'dir') == 7 && ~strcmp(pwd,nSTATRootDir)
# MATLAB:     cd(nSTATRootDir);
# MATLAB: end
#
# MATLAB: Direction=3; Neuron=1; Stim=2;
# MATLAB: datapath = fullfile(explicitStimulusDir,['Dir' num2str(Direction)], ...
# MATLAB:     ['Neuron' num2str(Neuron)],['Stim' num2str(Stim)]);
# MATLAB: data = load(fullfile(datapath,'trngdataBis.mat'));
#
# MATLAB: time=0:.001:(length(data.t)-1)*.001;
# MATLAB: stimData = data.t;
# MATLAB: spikeTimes = time(data.y==1);
#
# MATLAB: stim = Covariate(time,stimData./10,'Stimulus','time','s','mm',{'stim'});
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:     {'constant'});
#
# MATLAB: nst = nspikeTrain(spikeTimes);
# MATLAB: nspikeColl = nstColl(nst);
# MATLAB: cc = CovColl({stim,baseline});
# MATLAB: trial = Trial(nspikeColl,cc);
# trial.plot;
#
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
# MATLAB: subplot(3,1,1);
# MATLAB: nst2 = nspikeTrain(spikeTimes);
# MATLAB: nst2.setMaxTime(21);nst2.plot;
# MATLAB: set(gca,'ytick',[0 1]);
# MATLAB: xlabel('');
# MATLAB: hy=ylabel('spikes');
# MATLAB: set(hy,'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title({'Neural Raster'},'FontWeight','bold','FontSize',16,'FontName','Arial');
# MATLAB: set(gca, ...
# MATLAB:   'XTick'       , 0:1:max(time), ...
# MATLAB:   'XTickLabel'  , [],...
# MATLAB:   'LineWidth'   , 1         );
# MATLAB: subplot(3,1,2);
# MATLAB: stim.getSigInTimeWindow(0,21).plot([],{{' ''k'' '}}); legend off;
# MATLAB: set(gca,'ytick',[0 0.5 1]);
# MATLAB: hy=ylabel('Displacement [mm]','Interpreter','none'); xlabel('');
# MATLAB: set(hy,'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title({'Stimulus - Whisker Displacement'},'FontWeight','bold',...
# MATLAB:     'FontSize',16,'FontName','Arial');
#
# MATLAB: set(gca, ...
# MATLAB:   'XTick'       , 0:1:max(time), ...
# MATLAB:   'XTickLabel'  , [],...
# MATLAB:   'YTick'       , 0:.25:1, ...
# MATLAB:   'LineWidth'   , 1         );
#
# MATLAB: subplot(3,1,3);
# MATLAB: stim.derivative.getSigInTimeWindow(0,21).plot([],{{' ''k'' '}}); legend off;
# MATLAB: set(gca,'ytick',[-80 0 80]);
# MATLAB: axis([0 21 -80 80]);
# MATLAB: hy=ylabel('Displacement Velocity [mm/s]','Interpreter','none');
# MATLAB: hx= xlabel('time [s]','Interpreter','none');
# MATLAB: set([hx hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title({'Displacement Velocity'},'FontWeight','bold',...
# MATLAB:     'FontSize',16,'FontName','Arial');
#
# MATLAB: set(gca, ...
# MATLAB:   'XTick'       , 0:1:max(time), ...
# MATLAB:   'YTick'       , -80:40:80, ...
# MATLAB:   'LineWidth'   , 1         );
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 11

_ = section_index


In [ ]:
# MATLAB section 12/43 for nSTATPaperExamples: Section

# %
# Fit a constant baseline and Find Stimulus Lag
# We fit a constant rate (Poisson) model to the data and use the look at the
# cross-covariance function of between the stimulus and the fit
# residual to determine the appropriate lag for the stimulus.
# MATLAB: clear c; close all;
# MATLAB: selfHist = [] ; NeighborHist = []; sampleRate = 1000;
# MATLAB: c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
#
# Find Stimulus Lag (look for peaks in the cross-covariance function less
# than 1 second
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
#
# MATLAB: subplot(7,2,[1 3 5])
# MATLAB: results.Residual.xcov(stim).windowedSignal([0,1]).plot;
#
# MATLAB: ylabel('');
# MATLAB: [m,ind,ShiftTime] = max(results.Residual.xcov(stim).windowedSignal([0,1]));
# MATLAB: title(['Cross Correlation Function - Peak at t=' num2str(ShiftTime) ' sec'],'FontWeight','bold',...
# MATLAB:             'FontSize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB: hold on;
# MATLAB: h=plot(ShiftTime,m,'ro','Linewidth',3);
# MATLAB: set(h, 'MarkerFaceColor',[1 0 0], 'MarkerEdgeColor',[1 0 0]);
# MATLAB: hx=xlabel('Lag [s]','Interpreter','none');
# MATLAB: set(hx,'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
#
# Allow for shifts of less than 1 second
# MATLAB: stim = Covariate(time,stimData,'Stimulus','time','s','V',{'stim'});
# MATLAB: stim = stim.shift(ShiftTime);
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:     {'\mu'});
#
# MATLAB: nst = nspikeTrain(spikeTimes);
# MATLAB: nspikeColl = nstColl(nst);
# MATLAB: cc = CovColl({stim,baseline});
# MATLAB: trial2 = Trial(nspikeColl,cc);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 12

_ = section_index


In [ ]:
# MATLAB section 13/43 for nSTATPaperExamples: Compare constant rate model with model including stimulus effect

# % Compare constant rate model with model including stimulus effect
# Addition of the stimulus improves the fits in terms of the KS plot and
# the making the rescaled ISIs less correlated. The Point Process Residula
# also looks more "white"
# MATLAB: clear c;
# MATLAB: selfHist = [] ; NeighborHist = []; sampleRate = 1000;
# MATLAB: c{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,selfHist,...
# MATLAB:     NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: c{2} = TrialConfig({{'Baseline','\mu'},{'Stimulus','stim'}},...
# MATLAB:     sampleRate,selfHist,NeighborHist);
# MATLAB: c{2}.setName('Baseline+Stimulus');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial2,cfgColl,0);
# results.plotResults;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 13

_ = section_index


In [ ]:
# MATLAB section 14/43 for nSTATPaperExamples: History Effect

# % History Effect
# Determine the best history effect model using AIC, BIC, and KS statistic
# MATLAB: sampleRate=1000;
# MATLAB: delta=1/sampleRate*1;
# MATLAB: maxWindow=1; numWindows=32;
# MATLAB: windowTimes =unique(round([0 logspace(log10(delta),...
# MATLAB:     log10(maxWindow),numWindows)]*sampleRate)./sampleRate);
# MATLAB: results =Analysis.computeHistLagForAll(trial2,windowTimes,...
# MATLAB:     {{'Baseline','\mu'},{'Stimulus','stim'}},'BNLRCG',0,sampleRate,0);
#
# MATLAB: KSind = find(results{1}.KSStats.ks_stat == min(results{1}.KSStats.ks_stat));
# MATLAB: AICind = find((results{1}.AIC(2:end)-results{1}.AIC(1))== ...
# MATLAB:                min(results{1}.AIC(2:end)-results{1}.AIC(1))) +1;
# MATLAB: BICind = find((results{1}.BIC(2:end)-results{1}.BIC(1))== ...
# MATLAB:                min(results{1}.BIC(2:end)-results{1}.BIC(1))) +1;
# MATLAB: if(AICind==1)
# MATLAB:     AICind=inf;
# MATLAB: end
# MATLAB: if(BICind==1)
# MATLAB:     BICind=inf; %sometime BIC is non-decreasing and the index would be 1
# MATLAB: end
# MATLAB: windowIndex = min([AICind,BICind]) %use the minimum order model
# MATLAB: Summary = FitResSummary(results);
# Summary.plotSummary;
#
#
# MATLAB: clear c;
# MATLAB: if(windowIndex>1)
# MATLAB:     selfHist = windowTimes(1:windowIndex+1);
# MATLAB: else
# MATLAB:     selfHist = [];
# MATLAB: end
# MATLAB: NeighborHist = []; sampleRate = 1000;
#
# figure;
# MATLAB: subplot(7,2,2);
# MATLAB: x=0:length(windowTimes)-1;
# MATLAB: plot(x,results{1}.KSStats.ks_stat,'.-'); axis tight; hold on;
# MATLAB: plot(x(windowIndex),results{1}.KSStats.ks_stat(windowIndex),'r*');
#
# MATLAB:  set(gca,'XTick', 0:5:results{1}.numResults-1,'XTickLabel',[],...
# MATLAB:      'TickLength', [.02 .02] , ...
# MATLAB:   'XMinorTick', 'on','LineWidth'   , 1);
#
# MATLAB: hy=ylabel('KS Statistic');
# MATLAB: set(hy,'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: dAIC = results{1}.AIC-results{1}.AIC(1);
# MATLAB:  title({'Model Selection via change'; 'in KS Statistic, AIC, and BIC'},...
# MATLAB:      'FontWeight','bold',...
# MATLAB:             'FontSize',12,...
# MATLAB:             'FontName','Arial');
#
# MATLAB: subplot(7,2,4); plot(x,dAIC,'.-');
# MATLAB: set(gca,'XTick', 0:5:results{1}.numResults-1,'XTickLabel',[],...
# MATLAB:      'TickLength', [.02 .02] , ...
# MATLAB:   'XMinorTick', 'on','LineWidth'   , 1);
# MATLAB: hy=ylabel('\Delta AIC');axis tight; hold on;
# MATLAB: set(hy,'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: plot(x(windowIndex),dAIC(windowIndex),'r*');
# MATLAB: dBIC = results{1}.BIC-results{1}.BIC(1);
#
# MATLAB: subplot(7,2,6); plot(x,dBIC,'.-');
# MATLAB: hy=ylabel('\Delta BIC'); axis tight; hold on;
#
# MATLAB: plot(x(windowIndex),dBIC(windowIndex),'r*');
# MATLAB: hx=xlabel('# History Windows, Q');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: set(gca, ...
# MATLAB:   'TickLength'  , [.02 .02] , ...
# MATLAB:   'XMinorTick'  , 'on'      , ...
# MATLAB:   'XTick'       , 0:5:results{1}.numResults-1, ...
# MATLAB:   'LineWidth'   , 1         );
#
#
#
# Compare Baseline, Baseline+Stimulus Model, Baseline+History+Stimulus
# Addition of the history effect yields a model that falls within the 95%
# CI of the KS plot.
#
# MATLAB: c{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,[],NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: c{2} = TrialConfig({{'Baseline','\mu'},{'Stimulus','stim'}},...
# MATLAB:                     sampleRate,[],[]);
# MATLAB: c{2}.setName('Baseline+Stimulus');
# MATLAB: c{3} = TrialConfig({{'Baseline','\mu'},{'Stimulus','stim'}},...
# MATLAB:                     sampleRate,windowTimes(1:windowIndex),[]);
# MATLAB: c{3}.setName('Baseline+Stimulus+Hist');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial2,cfgColl,0);
# results.plotResults;
#
# MATLAB: results.lambda.setDataLabels({'\lambda_{const}','\lambda_{const+stim}',...
# MATLAB:     '\lambda_{const+stim+hist}'});
# MATLAB: subplot(7,2,[9 11 13]); results.KSPlot;
# MATLAB: subplot(7,2,[10 12 14]); results.plotCoeffs; legend off;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 14

_ = section_index


In [ ]:
# MATLAB section 15/43 for nSTATPaperExamples: Example 3 - PSTH Data

# % Example 3 - PSTH Data
#
# Generate a known Conditional Intensity Function
# We generated a known conditional intensity function (rate function) and
# generate distinct realizations of point processes consistent with this
# rate function. We use the method of thinning to simulate a point process.
# MATLAB: clear all;
# MATLAB: [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:     getPaperDataDirs();
# MATLAB: close all;
# MATLAB: delta = 0.001;
# MATLAB: Tmax = 1;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: f=2;
# MATLAB: mu = -3;
#
# MATLAB: tempData = 1*sin(2*pi*f*time)+mu; %lambda >=0
# MATLAB: lambdaData = exp(tempData)./(1+exp(tempData))*(1/delta);
# MATLAB: lambda = Covariate(time,lambdaData, '\lambda(t)','time','s',...
# MATLAB:     'spikes/sec',{'\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB: numRealizations = 20;
# MATLAB: spikeCollSim = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);
#
#
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
#
# MATLAB: subplot(2,2,3);spikeCollSim.plot;
# MATLAB: set(gca,'YTick',0:5:numRealizations,'YTickLabel',0:5:numRealizations);
# MATLAB: title({[num2str(numRealizations) ' Simulated Point Process Sample Paths']},...
# MATLAB:     'FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
#
# MATLAB: subplot(2,2,1);lambda.plot;
# MATLAB: title({'Simulated Conditional Intensity Function (CIF)'},...
# MATLAB:     'FontWeight','bold','FontSize',14,'FontName','Arial');
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: hy=get(gca,'YLabel');
# MATLAB: set(hy,'FontName', 'Arial','FontSize',14,'FontWeight','bold');
#
# MATLAB: x = load(fullfile(psthDir,'Results.mat'));
# MATLAB: numTrials = x.Results.Data.Spike_times_STC.balanced_SUA.Nr_trials;
# MATLAB: cellNum=6; clear nst;
# MATLAB: for i=1:numTrials
# MATLAB:     spikeTimes{i}=x.Results.Data.Spike_times_STC.balanced_SUA.spike_times{1,i,cellNum};
# MATLAB:     nst{i} = nspikeTrain(spikeTimes{i});
# MATLAB:     nst{i}.setName(num2str(cellNum));
# MATLAB: end
#
# MATLAB: spikeCollReal1=nstColl(nst);
# MATLAB: spikeCollReal1.setMinTime(0); spikeCollReal1.setMaxTime(2);
# MATLAB: subplot(2,2,2);spikeCollReal1.plot;  set(gca,'YTick',0:2:numTrials,...
# MATLAB:     'YTickLabel',0:2:numTrials);
# set(gca,'xtick',[0:.5:2],'xtickLabel',{'0','0.5','1','1.5','2'});
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: title('Response to Moving Visual Stimulus (Neuron 6)',...
# MATLAB:     'FontWeight','bold','Fontsize',14,'FontName','Arial');
#
# MATLAB: cellNum=1; clear nst;
# MATLAB: for i=1:numTrials
# MATLAB:     spikeTimes{i}=x.Results.Data.Spike_times_STC.balanced_SUA.spike_times{1,i,cellNum};
# MATLAB:     nst{i} = nspikeTrain(spikeTimes{i});
# MATLAB:     nst{i}.setName(num2str(cellNum));
# MATLAB: end
#
# MATLAB: spikeCollReal2=nstColl(nst);
# MATLAB: spikeCollReal2.setMinTime(0); spikeCollReal2.setMaxTime(2);
# MATLAB: subplot(2,2,4);spikeCollReal2.plot;
# MATLAB: set(gca,'YTick',0:2:numTrials,'YTickLabel',0:2:numTrials);
# set(gca,'xtick',[0:.5:2],'xtickLabel',{'0','0.5','1','1.5','2'});
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: title('Response to Moving Visual Stimulus (Neuron 1)','FontWeight',...
# MATLAB:     'bold','Fontsize',14,'FontName','Arial');
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 15

_ = section_index


In [ ]:
# MATLAB section 16/43 for nSTATPaperExamples: Estimate the PSTH with 50ms windows

# % Estimate the PSTH with 50ms windows
#
# MATLAB: close all;
#
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
#
# MATLAB: binsize = .05; %50ms window
# MATLAB: psth    = spikeCollSim.psth(binsize);
# MATLAB: psthGLM = spikeCollSim.psthGLM(binsize);
# MATLAB: true = lambda; %rate*delta = expected number of arrivals per bin
# MATLAB: subplot(2,3,4);
#
# MATLAB: h1=true.plot([],{{' ''b'',''Linewidth'',4'}});
# MATLAB: h3=psthGLM.plot([],{{' ''k'',''Linewidth'',4'}});
# MATLAB: h2=psth.plot([],{{' ''rx'',''Linewidth'',4'}});
#
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: ylabel('[spikes/sec]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
#
# MATLAB: legend off;
# MATLAB: h_legend=legend([h1(1) h2(1)  h3(1)],'true','PSTH','PSTH_{glm}');
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.005 pos(2)+.095 pos(3:4)]);
#
#
#
# MATLAB: subplot(2,3,1);spikeCollSim.plot;
# MATLAB: set(gca,'YTick',0:2:spikeCollSim.numSpikeTrains,'YTickLabel',0:2:spikeCollSim.numSpikeTrains);
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
#
# MATLAB: subplot(2,3,5);
# MATLAB: binsize = .05; %50ms window
# MATLAB: psthReal1    = spikeCollReal1.psth(binsize);
# MATLAB: psthGLMReal1 = spikeCollReal1.psthGLM(binsize);%,[],[],[],[],[],1000);
#
# MATLAB: h3=psthGLMReal1.plot([],{{' ''k'',''Linewidth'',4'}});
# MATLAB: h2=psthReal1.plot([],{{' ''rx'',''Linewidth'',4'}});
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('[spikes/sec]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
#
# MATLAB: h_legend=legend([h2(1)  h3(1)],'PSTH','PSTH_{glm}');
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.005 pos(2)+.07 pos(3:4)]);
# MATLAB: subplot(2,3,2); spikeCollReal1.plot;
# MATLAB: set(gca,'YTick',0:2:spikeCollReal2.numSpikeTrains,'YTickLabel',0:2:spikeCollReal2.numSpikeTrains);
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: subplot(2,3,6);
# MATLAB: psthReal2    = spikeCollReal2.psth(binsize);
# MATLAB: psthGLMReal2 = spikeCollReal2.psthGLM(binsize);%,[],[],[],[],[],1000);
# MATLAB: h3=psthGLMReal2.plot([],{{' ''k'',''Linewidth'',4'}});
# MATLAB: h2=psthReal2.plot([],{{' ''rx'',''Linewidth'',4'}});
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('[spikes/sec]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
#
#
# MATLAB: h_legend=legend([h2(1)  h3(1)],'PSTH','PSTH_{glm}');
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.005 pos(2)+.07 pos(3:4)]);
# MATLAB: subplot(2,3,3); spikeCollReal2.plot;
# MATLAB: set(gca,'YTick',0:2:spikeCollReal2.numSpikeTrains,'YTickLabel',0:2:spikeCollReal2.numSpikeTrains);
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 16

_ = section_index


In [ ]:
# MATLAB section 17/43 for nSTATPaperExamples: Example 3b - SSGLM Example

# % Example 3b - SSGLM Example
# Example of estimating with-in and across trial dynamics
# Methods from:
# G. Czanner, U. T. Eden, S. Wirth, M. Yanike,
# W. A. Suzuki, and E. N. Brown, "Analysis of between-trial and
# within-trial neural spiking dynamics.," Journal of neurophysiology,
# vol. 99, no. 5, pp. 2672?2693, May. 2008.
#
# MATLAB: close all;
# MATLAB: clear all;
# MATLAB: [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:     getPaperDataDirs();
# set(0,'DefaultFigureRenderer','ZBuffer')
# MATLAB: delta = 0.001; Tmax = 1;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: Ts=.001;
# MATLAB: numRealizations = 50; %Each realization corresponds to a distinct trial
#
# MATLAB: for i=1:numRealizations
# The within trial dynamics are sinusoidal
# For each trial the stimulus effect increases
# MATLAB:     f=2; b1(i)=3*((i)/numRealizations);b0=-3;
# MATLAB:     u = sin(2*pi*f*time);
# MATLAB:     e = zeros(length(time),1);   %No Ensemble input
#
# MATLAB:     stim=Covariate(time',u,'Stimulus','time','s','Voltage',{'sin'});
# MATLAB:     ens =Covariate(time',e,'Ensemble','time','s','Spikes',{'n1'});
#
# MATLAB:     mu=b0;
# MATLAB:     histCoeffs=[-4 -1 -.5];
# MATLAB:     H=tf(histCoeffs,[1],Ts,'Variable','z^-1');
#
# MATLAB:     S=tf([b1(i)],1,Ts,'Variable','z^-1');
# MATLAB:     E=tf([0],1,Ts,'Variable','z^-1');
# MATLAB:     simTypeSelect='binomial'; %Parameters are used to compute
# binomial conditional intensity function
#
#
# Obtain a realization of the point process with the current
# stimulus and history effect
# MATLAB:     [sC, lambdaTemp]=CIF.simulateCIF(mu,H,S,E,stim,ens,1,simTypeSelect);
#
# MATLAB:     if(i==1)
# MATLAB:         lambda=lambdaTemp; %Store the conditional intensity function
# MATLAB:     else
# MATLAB:         lambda = lambda.merge(lambdaTemp); %Add it to the other realizations
# MATLAB:     end
#
# MATLAB:     nst{i} = sC.getNST(1);             %get the neural spikeTrain from the collection
# MATLAB:     nst{i} = nst{i}.resample(1/delta); %make sure that it is sampled at the current samplerate
# MATLAB: end
#
# MATLAB: spikeColl = nstColl(nst); %Create a collection of the spike trains across trials
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 17

_ = section_index


In [ ]:
# MATLAB section 18/43 for nSTATPaperExamples: Summarize Simulated Data

# % Summarize Simulated Data
# MATLAB: close all;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
#
# Plot the raster
# MATLAB: subplot(3,2,[3 4]); spikeColl.plot;
# MATLAB: set(gca,'ytick',0:10:numRealizations,'ytickLabel',0:10:numRealizations);
# MATLAB: set(gca,'xtick',0:.1:Tmax,'xtickLabel',0:.1:Tmax); xlabel('');
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: title('Simulated Neural Raster','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',14,'FontWeight','bold');
#
# Plot the actual stimulus effect (not including history)
# The CIF including the history effect is stored in the lambda Covariate
# above
#
#
# MATLAB: stimData = exp(b0 + u'*b1);
# MATLAB: if(strcmp(simTypeSelect,'binomial'))
# MATLAB:     stimData = stimData./(1+stimData);
# MATLAB: end
#
# Plot the trial dependence
# MATLAB: subplot(3,2,1); plot(time,u,'k','LineWidth',3);
# xlabel('time [s]');ylabel('stimulus');
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Stimulus','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: title('Within Trial Stimulus','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',14,'FontWeight','bold');
#
# MATLAB: subplot(3,2,2); plot(1:length(b1),b1,'k','LineWidth',3);
# MATLAB: xlabel('Trial [k]','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: ylabel('Stimulus Gain','Interpreter','none','FontName', 'Arial','Fontsize',...
# MATLAB:     12,'FontWeight','bold');
# MATLAB: title('Across Trial Stimulus Gain','Interpreter','none','FontName',...
# MATLAB:     'Arial','Fontsize',14,'FontWeight','bold');
#
# MATLAB: subplot(3,2,[5 6]);
# MATLAB: imagesc(stimData'./delta);  set(gca, 'YDir','normal');
# MATLAB: set(gca,'xtick',0:100:Tmax/delta,'xtickLabel',0:.1:Tmax);
# MATLAB: set(gca,'ytick',0:10:numRealizations,'ytickLabel',0:10:numRealizations);
# MATLAB: xlabel('time [s]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: ylabel('Trial [k]','Interpreter','none','FontName', 'Arial',...
# MATLAB:     'Fontsize',12,'FontWeight','bold');
# MATLAB: title('True Conditional Intensity Function','Interpreter',...
# MATLAB:     'none','FontName', 'Arial','Fontsize',14,'FontWeight','bold');
#
#
# MATLAB: axis tight;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 18

_ = section_index


In [ ]:
# MATLAB section 19/43 for nSTATPaperExamples: Estimation of the Stimulus Response

# % Estimation of the Stimulus Response
#
# Create the covariates that will be used for the GLM regression
# MATLAB: stim = Covariate(time,sin(2*pi*f*time),'Stimulus','time','s','V',{'stim'});
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:                     {'constant'});
#
# Specify the windows of the history coefficients to be estimated
# MATLAB: windowTimes=[0:.001:.003];
# Number of bins to discrtize time into (used both for the PSTH and for
# thec
# SSGLM model.
# MATLAB: numBasis = 25;
#
# MATLAB: spikeColl.resample(1/delta); % Enforce sampleRate
# MATLAB: spikeColl.setMaxTime(Tmax);  % Make all spikeTrains end at time Tmax
#
#
# MATLAB: dN=spikeColl.dataToMatrix';  % Convert the spikeTrains into a matrix
# of 1's and 0's corresponding to the presence
# or absense of a spike in each time window.
# MATLAB: dN(dN>1)=1;                  % One should sample finely enough so there is
# one spike per bin. Here we make sure that
# this is the case regardless of the
# sampleRate
#
# The width of each rectangular basis pulse is determined by Tmax and by the
# number of basis pulses to use.
# MATLAB: basisWidth=(spikeColl.maxTime-spikeColl.minTime)/numBasis;
#
# MATLAB: if(simTypeSelect==0)
# MATLAB:     fitType='binomial';
# MATLAB: else
# MATLAB:     fitType='poisson';
# MATLAB: end
# MATLAB: if(strcmp(fitType,'binomial'))
# MATLAB:     Algorithm = 'BNLRCG';   % BNLRCG - faster Truncated, L-2 Regularized,
# Binomial Logistic Regression with Conjugate
# Gradient Solver by Demba Ba (demba@mit.edu).
# MATLAB: else
# MATLAB:     Algorithm = 'GLM';      % Standard Matlab GLM (Can be used for binomial or
# or Poisson CIFs
# MATLAB: end
#
# Use the values obtained from a PSTH to initialize the SSGLM filter
# MATLAB: [psthSig, ~, psthResult] =spikeColl.psthGLM(basisWidth,windowTimes,fitType);
# MATLAB: gamma0=psthResult.getHistCoeffs';%+.1*randn(size(histCoeffs));
# MATLAB: gamma0(isnan(gamma0))=-5; % Depending on the amount of data the
# the psth may not identify all parameters
# Just make sure that the estimates are real
# numbers
#
# MATLAB: x0=psthResult.getCoeffs;  %The initial estimate for the SSGLM model
#
# Estimate the variance within each time bin across trials
# MATLAB: numVarEstIter=10;
# MATLAB: Q0 = spikeColl.estimateVarianceAcrossTrials(numBasis,windowTimes,...
# MATLAB:     numVarEstIter,fitType);
# MATLAB: A=eye(numBasis,numBasis);
# MATLAB: delta = 1/spikeColl.sampleRate;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 19

_ = section_index


In [ ]:
# MATLAB section 20/43 for nSTATPaperExamples: Run the SSGLM Filter

# % Run the SSGLM Filter
# MATLAB: CompilingHelpFile=1;
# Commented out to speed up help file creation ...
# MATLAB:     if(~CompilingHelpFile)
# MATLAB:         Q0d=diag(Q0);
# MATLAB:         neuronName = psthResult.neuronNumber;
# MATLAB:         [xK,WK, WkuFinal,Qhat,gammahat,fitResults,stimulus,stimCIs,logll,...
# MATLAB:             QhatAll,gammahatAll,nIter]=DecodingAlgorithms.PPSS_EMFB(A,Q0d,x0,...
# MATLAB:             dN,fitType,delta,gamma0,windowTimes, numBasis,neuronName);
#
# MATLAB:         fR = fitResults.toStructure;
# MATLAB:         psthR = psthResult.toStructure;
# MATLAB:     end
# save SSGLMExampleData psthR fR xK WK WkuFinal Qhat gammahat fitResults stimulus stimCIs logll QhatAll gammahatAll nIter;

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 20

_ = section_index


In [ ]:
# MATLAB section 21/43 for nSTATPaperExamples: Section

# %
# MATLAB: installPath = which('nSTAT_Install');
# MATLAB: if isempty(installPath)
# MATLAB:     error('nSTATPaperExamples:MissingInstallPath', ...
# MATLAB:         'Could not locate nSTAT_Install.m on MATLAB path.');
# MATLAB: end
# MATLAB: nstatRoot = fileparts(installPath);
# MATLAB: if exist(nstatRoot,'dir') == 7 && ~strcmp(pwd,nstatRoot)
# MATLAB:     cd(nstatRoot);
# MATLAB: end
# MATLAB: addpath(nstatRoot,'-begin');
# MATLAB: load(fullfile(nstatRoot,'data','SSGLMExampleData.mat'));
# MATLAB: fitResults = FitResult.fromStructure(fR);
# MATLAB: psthResult = FitResult.fromStructure(psthR);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 21

_ = section_index


In [ ]:
# MATLAB section 22/43 for nSTATPaperExamples: Section

# %
# MATLAB: t=psthResult.mergeResults(fitResults);
# t.plotResults; %Compare the results with the PSTH Model
# MATLAB: t.lambda.setDataLabels({'\lambda_{PSTH}','\lambda_{SSGLM}'});
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
# MATLAB: subplot(2,2,1); t.KSPlot;
# MATLAB: subplot(2,2,2); t.plotResidual;
# MATLAB: subplot(2,2,3); t.plotInvGausTrans;
# MATLAB: subplot(2,2,4); t.plotSeqCorr;
#
# MATLAB: S=FitResSummary(t);
# MATLAB: dAIC=diff(S.AIC)
# MATLAB: dBIC=diff(S.BIC)
# MATLAB: dKS =diff(S.KSStats);
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 22

_ = section_index


In [ ]:
# MATLAB section 23/43 for nSTATPaperExamples: Section

# %
# MATLAB: close all;
# Generate the actual stimulus effect
# MATLAB: minTime=0; maxTime = Tmax;
# MATLAB: stimData = stim.data*b1;
# MATLAB: if(strcmp(fitType,'poisson'))
# MATLAB:     actStimEffect=exp(stimData + b0)./delta;
# MATLAB: elseif(strcmp(fitType,'binomial'))
# MATLAB:     actStimEffect=exp(stimData + b0)./(1+exp(stimData + b0))./delta;
# MATLAB: end
#
#
# Generate the basis function so that the estimated effect can be plotted
# at the same temporal resolution as the theoretical effect
# MATLAB:  if(~isempty(numBasis))
# MATLAB:     basisWidth = (maxTime-minTime)/numBasis;
# MATLAB:     sampleRate=1/delta;
# MATLAB:     unitPulseBasis=nstColl.generateUnitImpulseBasis(basisWidth,minTime,...
# MATLAB:         maxTime,sampleRate);
# MATLAB:     basisMat = unitPulseBasis.data;
# MATLAB:  end
#
# Generate the estimated stimulus effect
# MATLAB: if(strcmp(fitType,'poisson'))
# MATLAB:     estStimEffect=exp(basisMat*xK)./delta;
# MATLAB: elseif(strcmp(fitType,'binomial'))
# MATLAB:     estStimEffect=exp(basisMat*xK)./(1+exp(basisMat*xK))./delta;
# MATLAB: end
#
#
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.4 scrsz(4)*.8]);
#
# Plot the actual and estimated stimulus effect as a function of trial and
# time
# MATLAB: subplot(3,1,[1 2 3]);
# MATLAB: lighting gouraud
# MATLAB: surf((1:length(b1))',stim.time,actStimEffect,'FaceAlpha',0.1,...
# MATLAB:     'EdgeAlpha',0.1,'AlphaData',0.1);
# MATLAB: hx=xlabel('Trial [k]'); hy=ylabel('time [s]');
# MATLAB: hz=zlabel('Stimulus Effect [spikes/sec]'); hold all;
# MATLAB: set([hx hy hz],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# MATLAB: surf((1:length(b1))',stim.time,estStimEffect(:,1:length(b1)),...
# MATLAB:     'FaceAlpha',0.5,'EdgeAlpha',0.1,'AlphaData',0.5); %xlabel('Trial [k]'); ylabel('time [s]'); zlabel('Stimulus Effect');
# MATLAB: set(gca,'YDir','reverse');
# MATLAB: set(gca,'ytick',0:.1:Tmax,'ytickLabel',0:.1:Tmax);
#
# MATLAB: title('SSGLM Estimated vs. Actual Stimulus Effect','FontWeight','bold',...
# MATLAB:             'Fontsize',14,...
# MATLAB:             'FontName','Arial');
#
# MATLAB: close all;
# MATLAB: h=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.4 scrsz(4)*.8]);
#
# The actual stimulus effect
# MATLAB: subplot(3,1,1);
# MATLAB: lighting gouraud
# MATLAB: mesh((1:length(b1))',stim.time,actStimEffect);
# MATLAB: hx=xlabel('Trial [k]'); hy=ylabel('time [s]');
# MATLAB: zlabel('Stimulus Effect [spikes/sec]'); hold all;
# MATLAB: set([hx hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# title('True Stimulus Effect');
# MATLAB: title('True Stimulus Effect','FontWeight','bold',...
# MATLAB:             'Fontsize',14,...
# MATLAB:             'FontName','Arial');
# MATLAB: set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB: set(gca,'ytick',[],'ytickLabel',[]);
# MATLAB: CLIM = [min(min(stimData./delta)) max(max(stimData./delta))];
# MATLAB: view(gca,[90 -90]);
#
#
#
# The PSTH estimate
# MATLAB: subplot(3,1,2);
# MATLAB: lighting gouraud
# MATLAB: mesh((1:length(b1))',stim.time,repmat(psthSig.data, [1 numRealizations]));
# MATLAB: hx=xlabel('Trial [k]'); hy=ylabel('time [s]');
# MATLAB: hz=zlabel('Stimulus Effect [spikes/sec]'); hold all;
# MATLAB: set([hx hy hz],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# title('PSTH Estimated Stimulus Effect');
# MATLAB: title('PSTH Estimated Stimulus Effect','FontWeight','bold',...
# MATLAB:             'Fontsize',14,...
# MATLAB:             'FontName','Arial');
#
# MATLAB: set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB: set(gca,'ytick',[],'ytickLabel',[]);
# MATLAB: CLIM = [min(min(stimData./delta)) max(max(stimData./delta))];
# MATLAB: view(gca,[90 -90]);
#
# The SSGLM estimated stimulus effect
# MATLAB: subplot(3,1,3);
# MATLAB: lighting gouraud
# MATLAB: mesh((1:length(b1))',stim.time,estStimEffect);
# MATLAB: xlabel('Trial [k]'); ylabel('time [s]');
# MATLAB: zlabel('Stimulus Effect [spikes/sec]'); hold all;
# MATLAB: hx=get(gca,'XLabel');  hy=get(gca,'YLabel'); hz=get(gca,'ZLabel');
# MATLAB: set([hx hy hz],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# title('SSGLM Estimated Stimulus Efferct');
# MATLAB: title('SSGLM Estimated Stimulus Effect','FontWeight','bold',...
# MATLAB:             'Fontsize',14,...
# MATLAB:             'FontName','Arial');
# MATLAB: set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB: set(gca,'ytick',[],'ytickLabel',[]);
# MATLAB: set(gca, 'YDir','normal')
# MATLAB: view(gca,[90 -90]);
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 23

_ = section_index


In [ ]:
# MATLAB section 24/43 for nSTATPaperExamples: Compare differences across trials

# % Compare differences across trials
# MATLAB: echo off;
# MATLAB: close all;
# MATLAB:    minTime=0; maxTime = Tmax;
# Generate the basis function so that the estimated effect can be plotted
# at the same temporal resolution as the theoretical effect
# MATLAB:  if(~isempty(numBasis))
# MATLAB:     basisWidth = (maxTime-minTime)/numBasis;
# MATLAB:     sampleRate=1/delta;
# MATLAB:     unitPulseBasis=nstColl.generateUnitImpulseBasis(basisWidth,...
# MATLAB:         minTime,maxTime,sampleRate);
# MATLAB:     basisMat = unitPulseBasis.data;
# MATLAB:  end
#
#
# close all;
#
# MATLAB: t0=0; tf=Tmax;
# MATLAB: [spikeRateBinom, ProbMat,sigMat]=DecodingAlgorithms.computeSpikeRateCIs(xK,...
# MATLAB:     WkuFinal,dN,t0,tf,fitType,delta,gammahat,windowTimes);
#
# MATLAB: lt=find(sigMat(1,:)==1,1,'first');
# MATLAB: display(['The learning trial (compared to the first trial) is trial #' ...
# MATLAB:     num2str(find(sigMat(1,:)==1,1,'first'))]);
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.8]);
#
# MATLAB: subplot(2,3,1);
# MATLAB: spikeRateBinom.setName(['(' num2str(Tmax) '-0)^-1*\Lambda(0,' ...
# MATLAB:     num2str(Tmax) ')']);
# MATLAB: spikeRateBinom.plot([],{{' ''k'',''Linewidth'',4'}});
# e = Events(lt,{''});
# e.plot;
# MATLAB: v=axis;
# MATLAB: plot(lt*[1;1],v(3:4),'r','Linewidth',2);
# MATLAB: hx=xlabel('Trial [k]','Interpreter','none'); hold all;
# MATLAB: hy=ylabel('Average Firing Rate [spikes/sec]','Interpreter','none');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title(['Learning Trial:' num2str(lt)],'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
#
#
#
# MATLAB: h=subplot(2,3,[2 3 5 6]);
# MATLAB: K=size(dN,1);
# MATLAB: colormap(flipud(gray));
# MATLAB: imagesc(ProbMat); hold on;
# MATLAB: for k=1:K
# MATLAB:     for m=(k+1):K
# MATLAB:         if(sigMat(k,m)==1)
# MATLAB:             plot3(m,k,1,'r*'); hold on;
# MATLAB:         end
# MATLAB:     end
# MATLAB: end
#
# MATLAB: set(h,'XAxisLocation','top','YAxisLocation','right');
# MATLAB: hx=xlabel('Trial Number','Interpreter','none'); hold all;
# MATLAB: hy=ylabel('Trial Number','Interpreter','none');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# MATLAB: subplot(2,3,4)
# MATLAB: stim1 = Covariate(time, basisMat*stimulus(:,1),'Trial1','time','s',...
# MATLAB:     'spikes/sec');
# MATLAB: temp = ConfidenceInterval(time, basisMat*squeeze(stimCIs(:,1,:)));
# MATLAB: stim1.setConfInterval(temp);
# MATLAB: stimlt = Covariate(time, basisMat*stimulus(:,lt),'Trial1','time','s',...
# MATLAB:     'spikes/sec');
# MATLAB: temp = ConfidenceInterval(time, basisMat*squeeze(stimCIs(:,lt,:)));
# MATLAB: temp.setColor('r');
# MATLAB: stimlt.setConfInterval(temp);
# MATLAB: stimltm1 = Covariate(time, basisMat*stimulus(:,lt-1),'Trial1','time','s',...
# MATLAB:     'spikes/sec');
# MATLAB: temp = ConfidenceInterval(time, basisMat*squeeze(stimCIs(:,lt-1,:)));
# MATLAB: temp.setColor('r');
# MATLAB: stimltm1.setConfInterval(temp);
#
# figure;
# MATLAB: h1=stim1.plot([],{{' ''k'',''Linewidth'',4'}}); hold all;
# MATLAB: h2=stimlt.plot([],{{' ''r'',''Linewidth'',4'}});
# MATLAB: hx=xlabel('time [s]','Interpreter','none'); hold all;
# MATLAB: hy=ylabel('Firing Rate [spikes/sec]','Interpreter','none');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# MATLAB: title({'Learning Trial Vs. Baseline Trial';'with 95% CIs'},'FontWeight','bold',...
# MATLAB:             'Fontsize',12,...
# MATLAB:             'FontName','Arial');
# MATLAB: h_legend=legend([h1(1) h2(1)],'\lambda_{1}(t)',['\lambda_{' num2str(lt) '}(t)']);
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.03 pos(2)+.01 pos(3:4)]);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 24

_ = section_index


In [ ]:
# MATLAB section 25/43 for nSTATPaperExamples: Example 4 - HIPPOCAMPAL PLACE CELL - RECEPTIVE FIELD ESTIMATION

# % Example 4 - HIPPOCAMPAL PLACE CELL - RECEPTIVE FIELD ESTIMATION
# Estimation of receptive fields of neurons is a very common data analysis problem in neuroscience.
# Here we use the nSTAT software to perform an estimation of the receptive fields of hippocampal
# place cells using a bivariate Gaussian model and Zernike polynomials. The number of zernike polynomials
# is based on "An Analysis of Hippocampal Spatio-Temporal Representations Using a Bayesian Algorithm for Neural
# Spike Train Decoding" Barbieri et. al 2005. The data used herein in was
# provided by Dr. Ricardo Barbieri on 2/28/2011.
#
# *Author*: Iahn Cajigas
#
# *Date*: 3/1/2011

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 25

_ = section_index


In [ ]:
# MATLAB section 26/43 for nSTATPaperExamples: Section

# %
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 26

_ = section_index


In [ ]:
# MATLAB section 27/43 for nSTATPaperExamples: Example Data

# % Example Data
# The x and y coordinates of a freely foraging rat in a circular environment (70cm in diameter and 30cm high walls) and a fixed visual cue.
# The x and y coordinates at the time when a spike was observed are marked
# in red. The position coordinates have been normalized to be between -1
# and 1 to allow to simplify the analysis.
# MATLAB:     close all;
# MATLAB:     load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));
# MATLAB:     exampleCell = [2 21 25 49];
# exampleCell = 1:length(neuron);
# figure(1);
# MATLAB:     scrsz = get(0,'ScreenSize');
# MATLAB:     h=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.6 scrsz(4)*.9]);
#
# MATLAB:     for i=1:length(exampleCell)
# MATLAB:         subplot(2,2,i);
# MATLAB:         h1=plot(x,y,'b','Linewidth',.5); hold on;
# MATLAB:         h2=plot(neuron{exampleCell(i)}.xN,neuron{exampleCell(i)}.yN,'r.',...
# MATLAB:             'MarkerSize',7);
# MATLAB:         hx=xlabel('X Position'); hy=ylabel('Y Position');
# title(['Animal#1, Cell#' num2str(exampleCell(i))]);
# MATLAB:         title(['Cell#' num2str(exampleCell(i))],'FontWeight','bold',...
# MATLAB:             'Fontsize',12,'FontName','Arial');
# MATLAB:         set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:         set(gca,'xTick',-1:.5:1,'yTick',-1:.5:1); axis square;
# MATLAB:         if(i==4)
# MATLAB:             h_legend = legend([h1 h2],'Animal Path',...
# MATLAB:                 'Location at time of spike');
# MATLAB:             pos = get(h_legend,'position');
# MATLAB:             set(h_legend, 'position',[pos(1)+.09 pos(2)+.06 pos(3:4)]);
# MATLAB:         end
# MATLAB:     end
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 27

_ = section_index


In [ ]:
# MATLAB section 28/43 for nSTATPaperExamples: Analyze All Cells

# % Analyze All Cells
# MATLAB: numAnimals=2;
# MATLAB: CompilingHelpFile=1;
# MATLAB: if(~CompilingHelpFile)
# MATLAB:     for n=1:numAnimals
# load the data
# MATLAB:         clear x y neuron time nst tc tcc z;
# MATLAB:         load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));
#
# Create the spikeTrains for each cell
# MATLAB:         for i=1:length(neuron)
# MATLAB:             nst{i} = nspikeTrain(neuron{i}.spikeTimes);
# MATLAB:         end
#
#
# Convert to polar coordinates
# MATLAB:         [theta,r] = cart2pol(x,y);
#
#
# Evaluate the Zernike Polynomials
# Number of polynomials from "An Analysis of Hippocampal
# Spatio-Temporal Representations Using a Bayesian Algorithm for Neural
# Spike Train Decoding" Barbieri et. al 2005
# MATLAB:         cnt=0;
# MATLAB:         for l=0:3
# MATLAB:            for m=-l:l
# MATLAB:                if(~any(mod(l-m,2))) % otherwise the polynomial = 0
# MATLAB:                 cnt = cnt+1;
# MATLAB:                 z(:,cnt) = zernfun(l,m,r,theta,'norm');
# zernfun by Paul Fricker
# http://www.mathworks.com/matlabcentral/fileexchange/7687
# MATLAB:                end
# MATLAB:            end
# MATLAB:         end
#
# Data sampled at 30 Hz but just to be sure
# MATLAB:         delta=min(diff(time));
# MATLAB:         sampleRate = round(1/delta);
# Define Covariates for the analysis
# MATLAB:         baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',...
# MATLAB:                             {'mu'});
# MATLAB:         zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3',...
# MATLAB:                             'z4','z5','z6','z7','z8','z9','z10'});
# MATLAB:         gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time',...
# MATLAB:                             's','m',{'x','y','x^2','y^2','x*y'});
# MATLAB:         covarColl = CovColl({baseline,gaussian,zernike});
#
# Create the trial structure
# MATLAB:         spikeColl = nstColl(nst);
# MATLAB:         trial     = Trial(spikeColl,covarColl);
#
#
# Define how we want to analyze the data
# MATLAB:         tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian',...
# MATLAB:                             'x','y','x^2','y^2','x*y'}},sampleRate,[]);
# MATLAB:         tc{1}.setName('Gaussian');
# MATLAB:         tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6',...
# MATLAB:                             'z7','z8','z9','z10'}},sampleRate,[]);
# MATLAB:         tc{2}.setName('Zernike');
# MATLAB:         tcc = ConfigColl(tc);
#
# Perform Analysis (Commented to since data already saved)
# MATLAB:          results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
#
# Save results
# MATLAB:             resStruct =FitResult.CellArrayToStructure(results);
# MATLAB:             filename = fullfile(dataDir,['PlaceCellAnimal' num2str(n) 'Results']);
# MATLAB:             save(filename,'resStruct');
# MATLAB:     end
# MATLAB: end

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 28

_ = section_index


In [ ]:
# MATLAB section 29/43 for nSTATPaperExamples: View Summary Statistics

# % View Summary Statistics
# Note the Zernike Polynomials yield better fits in terms of decreased KS
# Statistics (less deviation from the 45 degree line), reduced AIC and
# reduced BIC across the majority of cells and for both animals
# MATLAB: clear Summary;
# MATLAB: numAnimals =2;
#
# MATLAB: for n=1:numAnimals
# MATLAB:     resData = load(fullfile(dataDir,['PlaceCellAnimal' num2str(n) 'Results.mat']));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
# MATLAB:     Summary{n} = FitResSummary(results);
# Summary{n}.plotSummary;
# MATLAB: end

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 29

_ = section_index


In [ ]:
# MATLAB section 30/43 for nSTATPaperExamples: Section

# %
# MATLAB: close all;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: h=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.6 scrsz(4)*.5]);
# MATLAB: subplot(1,3,1);
# MATLAB: maxLength = max([Summary{1}.numNeurons,Summary{2}.numNeurons]);
# MATLAB: dKS = nan(maxLength, 2);
# MATLAB: dKS(1:Summary{1}.numNeurons,1) = (Summary{1}.KSStats(:,1)-Summary{1}.KSStats(:,2)) ;
# MATLAB: dKS(1:Summary{2}.numNeurons,2) = (Summary{2}.KSStats(:,1)-Summary{2}.KSStats(:,2)) ;
#
# MATLAB: boxplot(dKS ,{'Animal 1', 'Animal 2'},'labelorientation','inline');
# MATLAB: title('\Delta KS Statistic','FontWeight','bold','FontSize',14,...
# MATLAB:     'FontName','Arial');
#
#
# MATLAB: subplot(1,3,2);
# MATLAB: dAIC = nan(maxLength, 2);
# MATLAB: dAIC(1:Summary{1}.numNeurons,1) = Summary{1}.getDiffAIC(1);
# MATLAB: dAIC(1:Summary{2}.numNeurons,2) = Summary{2}.getDiffAIC(1);
#
# MATLAB: boxplot(dAIC ,{'Animal 1', 'Animal 2'},'labelorientation','inline');
# MATLAB: title('\Delta AIC','FontWeight','bold','FontSize',14,'FontName','Arial');
#
#
# MATLAB: subplot(1,3,3);
# MATLAB: dBIC = nan(maxLength, 2);
# MATLAB: dBIC(1:Summary{1}.numNeurons,1) = Summary{1}.getDiffBIC(1);
# MATLAB: dBIC(1:Summary{2}.numNeurons,2) = Summary{2}.getDiffBIC(1);
#
# MATLAB: boxplot(dBIC ,{'Animal 1', 'Animal 2'},'labelorientation','inline'); %ylabel('\Delta BIC'); %xticklabel_rotate([],45,[],'Fontsize',6);
# MATLAB: title('\Delta BIC','FontWeight','bold','FontSize',14,'FontName','Arial');
#
# close all;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 30

_ = section_index


In [ ]:
# MATLAB section 31/43 for nSTATPaperExamples: Visualize the results

# % Visualize the results
# MATLAB: close all;
# Define a grid
# MATLAB: [x_new,y_new]=meshgrid(-1:.01:1); %define new x and y
# MATLAB: y_new = flipud(y_new); x_new = fliplr(x_new);
# MATLAB: [theta_new,r_new] = cart2pol(x_new,y_new);
#
# Data for the gaussian fit
# MATLAB: newData{1} =ones(size(x_new));
# MATLAB: newData{2} =x_new; newData{3} =y_new;
# MATLAB: newData{4} =x_new.^2; newData{5} =y_new.^2;
# MATLAB: newData{6} =x_new.*y_new;
#
#
# Zernike polynomials only defined on the unit disk
# MATLAB: idx = r_new<=1;
# MATLAB: zpoly = cell(1,10);
# MATLAB: cnt=0;
# MATLAB: for l=0:3
# MATLAB:    for m=-l:l
# MATLAB:        if(~any(mod(l-m,2)))
# MATLAB:         cnt = cnt+1;
# MATLAB:         temp = nan(size(x_new));
# MATLAB:         temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');
# MATLAB:         zpoly{cnt} = temp;
# MATLAB:        end
# MATLAB:    end
# MATLAB: end
#
#
#
# MATLAB: for n=1:numAnimals
#
# MATLAB:     clear lambdaGaussian lambdaZernike;
# MATLAB:     load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));
# MATLAB:     resData = load(fullfile(dataDir,['PlaceCellAnimal' num2str(n) 'Results.mat']));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
#
# MATLAB:     for i=1:length(neuron)
# Evaluate our fits using the new data and the estimated parameters
# MATLAB:         lambdaGaussian{i} = results{i}.evalLambda(1,newData);
# MATLAB:         lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);
# MATLAB:     end
#
#
#
#
# Plot the receptive fields
# MATLAB:     for i=1:length(neuron)
# 3d plot of an example place field
#
#
# 2d plot of all the cell's fields
# MATLAB:         if(n==1)
# MATLAB:             h4=figure(4);
# MATLAB:             colormap('jet');
# MATLAB:             if(i==1)
# MATLAB:                 tb=annotation(h4,'textbox',...
# MATLAB:                     [0.283261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Gaussian Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on','Fontsize',11,...
# MATLAB:                     'FontName','Arial','FontWeight','bold','LineStyle',...
# MATLAB:                     'none','HorizontalAlignment','center'); hold on;
# MATLAB:             end
# MATLAB:             subplot(7,7,i);
# MATLAB:         elseif(n==2)
# MATLAB:             h6=figure(6);
# MATLAB:             colormap('jet');
# MATLAB:             if(i==1)
# MATLAB:                 annotation(h6,'textbox',...
# MATLAB:                     [0.283261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Gaussian Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on','Fontsize',11,...
# MATLAB:                     'FontName','Arial','FontWeight','bold','LineStyle',...
# MATLAB:                     'none','HorizontalAlignment','center'); hold on;
# MATLAB:             end
# MATLAB:             subplot(6,7,i);
# MATLAB:         end
# MATLAB:         pcolor(x_new,y_new,lambdaGaussian{i}), shading interp
# MATLAB:         axis square; set(gca,'xtick',[],'ytick',[]);
# MATLAB:         set(gca, 'Box'         , 'off');
#
# MATLAB:         if(n==1)
# MATLAB:             h5=figure(5);
# MATLAB:             colormap('jet');
# MATLAB:             if(i==1)
# MATLAB:                 annotation(h5,'textbox',...
# MATLAB:                     [0.303261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Zernike Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on','Fontsize',11,...
# MATLAB:                     'FontName','Arial','FontWeight','bold','LineStyle','none'); hold on;
#
# MATLAB:             end
# MATLAB:             subplot(7,7,i);
# MATLAB:         elseif(n==2)
# MATLAB:             h7=figure(7);
# MATLAB:             colormap('jet');
# MATLAB:             if(i==1)
# MATLAB:                annotation(h7,'textbox',...
# MATLAB:                     [0.303261904761904 0.928571428571418 ...
# MATLAB:                     0.392857142857143 0.0595238095238095],...
# MATLAB:                     'String',{['Zernike Place Fields - Animal#' ...
# MATLAB:                     num2str(n)]},'FitBoxToText','on','Fontsize',11,...
# MATLAB:                     'FontName','Arial','FontWeight','bold','LineStyle',...
# MATLAB:                     'none','HorizontalAlignment','center'); hold on;
# MATLAB:             end
# MATLAB:             subplot(6,7,i);
# MATLAB:         end
# MATLAB:         pcolor(x_new,y_new,lambdaZernike{i}), shading interp
# MATLAB:         axis square;
# MATLAB:         set(gca,'xtick',[],'ytick',[]);
# MATLAB:         set(gca, 'Box'         , 'off');
# MATLAB:     end
#
#
# MATLAB: end
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 31

_ = section_index


In [ ]:
# MATLAB section 32/43 for nSTATPaperExamples: Section

# %
# MATLAB:     clear lambdaGaussian lambdaZernike;
# MATLAB:     load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));
# MATLAB:     resData = load(fullfile(dataDir,'PlaceCellAnimal1Results.mat'));
# MATLAB:     results = FitResult.fromStructure(resData.resStruct);
#
# MATLAB:     for i=1:length(neuron)
# Evaluate our fits using the new data and the estimated parameters
# MATLAB:         lambdaGaussian{i} = results{i}.evalLambda(1,newData);
# MATLAB:         lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);
# MATLAB:     end
#
#
#
# h1=plot(x,y,'b');
# h2=plot(x,y,'g');
#
# MATLAB:     exampleCell = 25;
# figure(8);
# plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');
# xlabel('x'); ylabel('y');
# title(['Animal#1, Cell#' num2str(exampleCell)]);
#
# MATLAB:     close all;
# MATLAB:     h9=figure(9);
# MATLAB:     h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);
# MATLAB:     get(h_mesh,'AlphaData');
# MATLAB:     set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','b');
# MATLAB:     hold on;
# MATLAB:     h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);
# MATLAB:     get(h_mesh,'AlphaData');
# MATLAB:     set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','g');
#
#
# h_legend=legend('\lambda_{Gaussian}','\lambda_{Zernike}');
# set(h_legend,'FontSize',20);
# MATLAB:     plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');
# MATLAB:     axis tight square;
# MATLAB:     xlabel('x position'); ylabel('y position');
# MATLAB:     title(['Animal#1, Cell#' num2str(exampleCell)],'FontWeight','bold',...
# MATLAB:         'Fontsize',12,'FontName','Arial');
# MATLAB:     hx=get(gca,'XLabel');  hy=get(gca,'YLabel');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 32

_ = section_index


In [ ]:
# MATLAB section 33/43 for nSTATPaperExamples: Example 5 - STIMULUS DECODING

# % Example 5 - STIMULUS DECODING
# In this example we show how to decode a univariate and a bivariate
# stimulus based on a point process observations using nSTAT. Even though
# due to the simulated nature of the data, we know the exact condition
# intensity function, we estimate the parameters before moving on to the
# decoding stage.

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 33

_ = section_index


In [ ]:
# MATLAB section 34/43 for nSTATPaperExamples: Generate the conditional Intensity Function

# % Generate the conditional Intensity Function
#
# MATLAB:     close all; clear all;
# MATLAB:     [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:         getPaperDataDirs();
# MATLAB:     delta = 0.001; Tmax = 1;
# MATLAB:     time = 0:delta:Tmax;
# MATLAB:     numRealizations = 20;
# MATLAB:     f=2; b1=randn(numRealizations,1);b0=log(10*delta)+randn(numRealizations,1);
# MATLAB:     x = sin(2*pi*f*time);
# MATLAB:     clear nst;
# MATLAB:     for i=1:numRealizations
# MATLAB:         expData = exp(b1(i)*x+b0(i));
# MATLAB:         lambdaData = expData./(1+expData);
#
# MATLAB:         if(i==1)
# MATLAB:             lambda = Covariate(time,lambdaData./delta, ...
# MATLAB:                 '\Lambda(t)','time','s','spikes/sec',{'\lambda_{1}'},...
# MATLAB:                 {{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:         else
# MATLAB:             tempLambda = Covariate(time,lambdaData./delta, ...
# MATLAB:                 '\Lambda(t)','time','s','spikes/sec',{'\lambda_{1}'},...
# MATLAB:                 {{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:             lambda = lambda.merge(tempLambda);
# MATLAB:         end
#
# MATLAB:         spikeColl = CIF.simulateCIFByThinningFromLambda(...
# MATLAB:             lambda.getSubSignal(i),1);
# MATLAB:         nst{i} = spikeColl.getNST(1);
# MATLAB:     end
# MATLAB:         spikeColl = nstColl(nst);scrsz = get(0,'ScreenSize');
# MATLAB:         h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:             scrsz(3)*.6 scrsz(4)*.8]);
# figure;
# MATLAB:         subplot(3,1,1); plot(time,x,'k');
# MATLAB:         set(gca,'xtick',[],'xtickLabel',[]); ylabel('Stimulus');
# MATLAB:             hx=get(gca,'XLabel');  hy=get(gca,'YLabel');
# MATLAB:             set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:             title('Driving Stimulus','FontWeight','bold',...
# MATLAB:                 'FontSize',14,'FontName','Arial');
# MATLAB:         subplot(3,1,2); lambda.plot([],{{' ''k'',''Linewidth'',1'}});
# MATLAB:             legend off;
# MATLAB:             hy=ylabel('Firing Rate [spikes/sec]', 'Interpreter','none');
# MATLAB:             hx=xlabel('','Interpreter','none');
# MATLAB:             set([hx, hy],'FontName', 'Arial','FontSize',12,...
# MATLAB:                 'FontWeight','bold');
# MATLAB:             set(gca,'xtickLabel',[]);
# MATLAB:             title('Conditional Intensity Functions','FontWeight',...
# MATLAB:                 'bold','FontSize',14,'FontName','Arial');
#
# MATLAB:         subplot(3,1,3); spikeColl.plot;
# MATLAB:             set(gca,'ytick',0:10:numRealizations,'ytickLabel',...
# MATLAB:                 0:10:numRealizations);
# MATLAB:             xlabel('time [s]','Interpreter','none');
# MATLAB:             ylabel('Cell Number','Interpreter','none');
# MATLAB:             hx=get(gca,'XLabel');  hy=get(gca,'YLabel');
# MATLAB:             set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:             title('Point Process Sample Paths','FontWeight',...
# MATLAB:                 'bold','FontSize',14,'FontName','Arial');
#
# MATLAB: stim = Covariate(time,sin(2*pi*f*time),'Stimulus','time','s','V',{'stim'});
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:                     {'constant'});
#
# close all;

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 34

_ = section_index


In [ ]:
# MATLAB section 35/43 for nSTATPaperExamples: Section

# %
# MATLAB: close all;
#
# MATLAB: clear lambdaCIF;
# MATLAB: spikeColl.resample(1/delta);
# MATLAB: dN=spikeColl.dataToMatrix;
#
# Make noise according to the dynamic range of the stimulus
# MATLAB: Q=std(stim.data(2:end)-stim.data(1:end-1));
# MATLAB: Px0=.1; A=1;
# MATLAB: x0 = x(:,1); yT=x(:,end);
# MATLAB: Pi0 = eps*eye(size(x0,1),size(x0,1));
# MATLAB: PiT = eps*eye(size(x0,1),size(x0,1));
#
#
# MATLAB: [x_p, W_p, x_u, W_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, ...
# MATLAB:     Q, dN',b0,b1','binomial',delta);
#
# MATLAB: h=figure('Position',[scrsz(3)*.1 scrsz(4)*.1 scrsz(3)*.8 scrsz(4)*.6]);
# MATLAB: zVal=1.96;
# MATLAB: ciLower = min(x_u(1:end)-zVal*sqrt(squeeze(W_u(1:end)))',...
# MATLAB:     x_u(1:end)+zVal*sqrt(squeeze(W_u(1:end))'));
# MATLAB: ciUpper = max(x_u(1:end)-zVal*sqrt(squeeze(W_u(1:end)))',...
# MATLAB:     x_u(1:end)+zVal*sqrt(squeeze(W_u(1:end))'));
#
# MATLAB: estimatedStimulus = Covariate(time,x_u(1:end),'\hat{x}(t)','time','s','');
# MATLAB: CI= ConfidenceInterval(time,[ciLower', ciUpper'],'\hat{x}(t)','time','s','');
# MATLAB: estimatedStimulus.setConfInterval(CI);
#
# hold all;
# hEst=plot(time,x_u(1:end),'b','Linewidth',2); hold on;
# plot(time, [ciUpper', ciLower'],'b');
#
# MATLAB: hEst = estimatedStimulus.plot([],{{' ''k'',''Linewidth'',4'}});
# MATLAB: hStim=stim.plot([],{{' ''b'',''Linewidth'',4'}});
# MATLAB: legend off;
# MATLAB: h_legend=legend([hEst(1) hStim],'Decoded','Actual');
# MATLAB: set(h_legend,'Interpreter','none');
# MATLAB: set(h_legend,'FontSize',22);
# MATLAB: title(['Decoded Stimulus +/- 95% CIs with ' num2str(numRealizations) ' cells'],...
# MATLAB:     'FontWeight','bold','Fontsize',22,'FontName','Arial');
# MATLAB: xlabel('time [s]','Interpreter','none');
# MATLAB: ylabel('Stimulus','Interpreter','none');
# MATLAB: hx=get(gca,'XLabel');  hy=get(gca,'YLabel');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',22,'FontWeight','bold');
#
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 35

_ = section_index


In [ ]:
# MATLAB section 36/43 for nSTATPaperExamples: Example 5b - Arm reaching to target Simulation

# % Example 5b - Arm reaching to target Simulation
# See
# L. Srinivasan, U. T. Eden, A. S. Willsky, and E. N. Brown,
# "A state-space analysis for reconstruction of goal-directed movements
# using neural signals.," Neural computation, vol. 18, no. 10, pp. 2465?2494, Oct. 2006.
#
# MATLAB:     close all;
# MATLAB:     clear all;
# MATLAB:     [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:         getPaperDataDirs();
# Process noise covariance only drives the movement velocity
# MATLAB:     q=1e-4;
# MATLAB:     Q=diag([1e-12 1e-12 q q]);
#
# MATLAB:     delta = .001;        % Time increment
# MATLAB:     r=1e-6;   % in meters^2
# MATLAB:     p=1e-6;    % in meters^2/s^2
# MATLAB:     PiT=diag([r r p p]); % Uncertainty in the target state
# MATLAB:     Pi0=PiT;
# MATLAB:     T=2;                 % Reach Duration
#
# MATLAB:     x0 = [0;0;0;0];     % Initial Position and velocities (states)
# MATLAB:     xT = [-.35;.2; 0;0];% Final Target
# MATLAB:     time=0:delta:T;     % time vector
#
# MATLAB:     A=[1 0 delta 0    ; %State transition matrix
# MATLAB:        0 1 0     delta;
# MATLAB:        0 0 1     0    ;
# MATLAB:        0 0 0     1    ];
#
# MATLAB:     x=zeros(4,length(time));
#
#
# Simulate a reach trajectory
# Differs from reference by multiplication by delta instead of division so
# that the velocity has units of meters per second
# MATLAB:     R=chol(Q);
# MATLAB:     L=chol(PiT);
# MATLAB:     for k=1:length(time)
# MATLAB:         if(k==1)
# MATLAB:             x(:,k)=x0;
# MATLAB:         else
# MATLAB:              x(:,k)=A*x(:,k-1)+...
# MATLAB:                  delta/(2)*(pi/T)^2*cos(pi*time(k)/T)*[0;0;...
# MATLAB:                  xT(1)-x0(1);xT(2)-x0(2)]; %Reach to target model
# x(:,k)=A*x(:,k-1)+R*randn(size(x,1),1); %Random walk model
# MATLAB:         end
#
# MATLAB:     end
# MATLAB:     xT =x(:,end); % The target generated by the model
# MATLAB:     yT=xT;        % Assume we have observed the actual target position with uncertainty PiT
#
# Define Q according to the dynamic range of the movement above
# MATLAB:     Q=diag(var(diff(x,[],2),[],2))*100;
#
# Plot the movement trajectories and the hand path
# MATLAB:     scrsz = get(0,'ScreenSize');
# MATLAB:     fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:         scrsz(3)*.8 scrsz(4)*.8]);
# Plot The movement path
# MATLAB:     subplot(4,2,[1 3]);
# MATLAB:     plot(100*x(1,:),100*x(2,:),'k','Linewidth',2);
# MATLAB:     xlabel('X Position [cm]'); ylabel('Y Position [cm]');
# MATLAB:     hx=get(gca,'XLabel');  hy=get(gca,'YLabel');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:     title('Reach Path','FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB:     hold on;
# MATLAB:     axis([sort([100*x0(1)+5, 100*xT(1)-5]), sort([100*x0(2)-5, 100*xT(2)+5])]);
# MATLAB:     h1=plot(100*x(1,1),100*x(2,1),'bo','MarkerSize',14);
# MATLAB:     h2=plot(100*x(1,end),100*x(2,end),'ro','MarkerSize',14);
# MATLAB:     legend([h1 h2],'Start','Finish','Location','NorthEast');
#
#
# MATLAB:     subplot(4,2,5); h1=plot(time,100*x(1,:),'k','Linewidth',2); hold on;
# MATLAB:     h2=plot(time,100*x(2,:),'k-.','Linewidth',2);
# MATLAB:     h_legend=legend([h1,h2],'x','y','Location','NorthEast');
# MATLAB:     set(h_legend,'FontSize',14)
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
# MATLAB:     hx=xlabel('time [s]'); hy=ylabel('Position [cm]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# Plot the velocity profiles
#
# MATLAB:     subplot(4,2,7);
# MATLAB:     h1=plot(time,100*x(3,:),'k','Linewidth',2); hold on;
# MATLAB:     h2=plot(time,100*x(4,:),'k-.','Linewidth',2);
# MATLAB:     h_legend=legend([h1 h2],'v_x','v_y','Location','NorthEast');
# MATLAB:     xlabel('time [s]');
# MATLAB:     set(h_legend,'FontSize',14);
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
# MATLAB:     hx=xlabel('time [s]'); hy=ylabel('Velocity [cm/s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
#
# MATLAB:     gamma=0;
# MATLAB:     windowTimes=[0, 0.001];
#
#
# Simulate neural responses
# logit(lambda_i*delta) = mu_i + b_x_i*v_x + b_y_i*v_y
# logit(lambda_i*delta) = X_i*beta_i;
# MATLAB:     numCells = 20;
# MATLAB:     bCoeffs=10*(rand(numCells,2)-.5);           % b_i = [b_x_i b_y_i] ~ U(-5, 5);
# MATLAB:     phiMax = atan2(bCoeffs(:,2),bCoeffs(:,1));  % Maximal firing direction of cell
# MATLAB:     phiMaxNorm = (phiMax+pi)./(2*pi);
# MATLAB:     meanMu = log(10*delta); % baseline firing rate -10Hz
# MATLAB:     MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
#
# MATLAB:     dataMat = [ones(length(time),1) x(3,:)' x(4,:)']; % design matrix: X (
# MATLAB:     coeffs = [MuCoeffs bCoeffs]; % coefficient vector: beta
# MATLAB:     fitType='binomial';
# MATLAB:     clear nst;
# MATLAB:     for i=1:numCells
# MATLAB:          tempData  = exp(dataMat*coeffs(i,:)');
#
# MATLAB:          if(strcmp(fitType,'poisson'))
# MATLAB:              lambdaData = tempData;
# MATLAB:          else
# MATLAB:             lambdaData = tempData./(1+tempData); % Conditional Intensity Function for ith cell
# MATLAB:          end
# MATLAB:          lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:              '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:              {strcat('\lambda_{',num2str(i),'}')},{{' ''b'' '}});
# MATLAB:          lambda{i}=lambda{i}.resample(1/delta);
#
# Generate CIF representation in case we want to use the symbolic
# versions of the PPDecodeFilter (i.e. not PPDecodeFilterLinear
# MATLAB:          lambdaCIF{i} = CIF([MuCoeffs(i) 0 0 bCoeffs(i,:)],...
# MATLAB:              {'1','x','y','vx','vy'},{'x','y','vx','vy'},fitType);
# generate one realization for each cell
# MATLAB:          tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1);          nst{i} = tempSpikeColl{i}.getNST(1);     % grab the realization
# MATLAB:          nst{i}.setName(num2str(i));              % give each cell a unique name
# MATLAB:          subplot(4,2,[6 8]);
# MATLAB:          h2=lambda{i}.plot([],{{' ''k'', ''LineWidth'' ,.5'}});
# MATLAB:          legend off; hold all; % Plot the CIF
#
#
#
# MATLAB:     end
# MATLAB:     title('Neural Conditional Intensity Functions','FontWeight',...
# MATLAB:         'bold','Fontsize',14,'FontName','Arial');
# MATLAB:     hx=xlabel('time [s]','Interpreter','none');
# MATLAB:     hy=ylabel('Firing Rate [spikes/sec]','Interpreter','none');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB:     spikeColl = nstColl(nst); % Create a neural spike train collection
#
# MATLAB:     subplot(4,2,[2,4]); spikeColl.plot;
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     title('Neural Raster','FontWeight','bold','Fontsize',14,...
# MATLAB:         'FontName','Arial');
# MATLAB:     hx=xlabel('time [s]','Interpreter','none');
# MATLAB:     hy=ylabel('Cell Number','Interpreter','none');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# close all;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 36

_ = section_index


In [ ]:
# MATLAB section 37/43 for nSTATPaperExamples: Section

# %
# MATLAB: close all;
# MATLAB: numExamples=20;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:     scrsz(3)*.6 scrsz(4)*.9]);
# MATLAB: for k=1:numExamples
# MATLAB:      bCoeffs=10*(rand(numCells,2)-.5);           % b_i = [b_x_i b_y_i] ~ U(-5, 5);
# MATLAB:     phiMax = atan2(bCoeffs(:,2),bCoeffs(:,1));  % Maximal firing direction of cell
# MATLAB:     phiMaxNorm = (phiMax+pi)./(2*pi);
# MATLAB:     meanMu = log(10*delta);  % baseline firing rate
# MATLAB:     MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
#
# MATLAB:     dataMat = [ones(length(time),1) x(3,:)' x(4,:)']; % design matrix: X (
# MATLAB:     coeffs = [MuCoeffs bCoeffs]; % coefficient vector: beta
# MATLAB:     fitType='binomial';
# MATLAB:     clear nst lambda;
#
#
# MATLAB:     for i=1:numCells
# MATLAB:         tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:          if(strcmp(fitType,'poisson'))
# MATLAB:             lambdaData = tempData;
# MATLAB:          else
# Conditional Intensity Function for ith cell
# MATLAB:             lambdaData = tempData./(1+tempData);
# MATLAB:          end
# MATLAB:          lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:              '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:              {strcat('\lambda_{',num2str(i),'}')},{{' ''b'' '}});
# MATLAB:          lambda{i}=lambda{i}.resample(1/delta);
#
# Generate CIF representation in case we want to use the symbolic
# versions of the PPDecodeFilter (i.e. not PPDecodeFilterLinear
# generate one realization for each cell
# MATLAB:          tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1);
# MATLAB:          nst{i} = tempSpikeColl{i}.getNST(1);     % grab the realization
# MATLAB:          nst{i}.setName(num2str(i));              % give each cell a unique name
#
# MATLAB:     end
#
# Plot the neural raster across all the cells
# MATLAB:     spikeColl = nstColl(nst); % Create a neural spike train collection
#
# Based on the temporal resolution defined by delta, bin the data and get
# a matrix representation of the neural firing
# MATLAB:     dN=spikeColl.dataToMatrix';
# MATLAB:     dN(dN>1)=1; % more than one spike per bin will be treated as one spike. In
# general we should pick delta small enough so that there is
# only one spike per bin
#
# MATLAB:     [C,N]   = size(dN); % N time samples, C cells
#
# MATLAB:     beta=[zeros(2,numCells);  bCoeffs'];
#
#
# Use the Goal Directed Movement Version of the Point Process adaptive
# Filter
# MATLAB:     [x_p, W_p, x_u, W_u,x_uT,W_uT,x_pT,W_pT] = ...
# MATLAB:         DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN,...
# MATLAB:         MuCoeffs,beta,fitType,delta,gamma,windowTimes,x0, Pi0, yT,PiT,0);
#
# Use the Free Movement Version of the Point Process adaptive
# Filter
# MATLAB:     [x_pf, W_pf, x_uf, W_uf] = ...
# MATLAB:         DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN,...
# MATLAB:         MuCoeffs,beta,fitType,delta,gamma,windowTimes,x0);
#
#
# MATLAB:     if(k==numExamples)
# MATLAB:         subplot(4,2,1:4);h1=plot(100*x(1,:),100*x(2,:),'k','LineWidth',3);
# MATLAB:         hold on;
# MATLAB:         axis([sort([100*x0(1)+5, 100*xT(1)-5]), ...
# MATLAB:             sort([100*x0(2)-5, 100*xT(2)+5])]);
# MATLAB:         title('Estimated vs. Actual Reach Paths',...
# MATLAB:             'FontWeight','bold','Fontsize',12,'FontName','Arial');
# MATLAB:     end
# MATLAB:     subplot(4,2,1:4);h2=plot(100*x_u(1,:)',100*x_u(2,:)','b'); hold all;
# MATLAB:     subplot(4,2,1:4);h3=plot(100*x_uf(1,:)',100*x_uf(2,:)','g');
# MATLAB:     hx=xlabel('x [cm]'); hy=ylabel('y [cm]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     h1=plot(100*x0(1),100*x0(2),'bo','MarkerSize',10); hold on;
# MATLAB:     h2=plot(100*xT(1),100*xT(2),'ro','MarkerSize',10);
# MATLAB:     legend([h1 h2],'Start','Finish','Location','NorthEast');
#
#
# MATLAB:     subplot(4,2,5);
# MATLAB:     h1=plot(time,100*x(1,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*x_u(1,:)','b');
# MATLAB:     h3=plot(time,100*x_uf(1,:)','g');
# MATLAB:     hy=ylabel('x(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# MATLAB:     subplot(4,2,6);
# MATLAB:     h1=plot(time,100*x(2,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*x_u(2,:)','b');
# MATLAB:     h3=plot(time,100*x_uf(2,:)','g');
# MATLAB:     h_legend=legend([h1(1) h2(1) h3(1)],'Actual','PPAF+Goal',...
# MATLAB:         'PPAF','Location','SouthEast');
# MATLAB:     hy=ylabel('y(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
# MATLAB:     set(h_legend,'FontSize',10)
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)-.63 pos(2)+.23 pos(3:4)]);
#
# MATLAB:     subplot(4,2,7);
# MATLAB:     h1=plot(time,100*x(3,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*x_u(3,:)','b');
# MATLAB:     h3=plot(time,100*x_uf(3,:)','g');
# MATLAB:     hy=ylabel('v_{x}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# MATLAB:     subplot(4,2,8);
# MATLAB:     h1=plot(time,100*x(4,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*x_u(4,:)','b');
# MATLAB:     h3=plot(time,100*x_uf(4,:)','g');
# MATLAB:     hy=ylabel('v_{y}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
#
# MATLAB: end
#
# close all;

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 37

_ = section_index


In [ ]:
# MATLAB section 38/43 for nSTATPaperExamples: Experiment 6 - Hybrid Point Process Filter Example

# % Experiment 6 - Hybrid Point Process Filter Example
# NOTE THIS EXAMPLE WAS NOT INCLUDED IN THE FINAL VERSION OF THE PAPER
# This example is based on an implementation of the Hybrid Point Process
# filter described in _General-purpose filter design for neural prosthetic
# devices_ by Srinivasan L, Eden UT, Mitter SK, Brown EN in J Neurophysiol.
# 2007 Oct, 98(4):2456-75.
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 38

_ = section_index


In [ ]:
# MATLAB section 39/43 for nSTATPaperExamples: Problem Statement

# % Problem Statement
# Suppose that a process of interest can be modeled as consisting of
# several discrete states where the evolution of the system under each
# state can be modeled as a linear state space model. The observations of
# both the state and the continuous dynamics are not direct, but rather
# observed through how the continuous and discrete states affect the firing
# of a population of neurons. The goal of the hybrid filter is to estimate
# both the continuous dynamics and the underlying system state from only
# the neural population firing (point process observations).
#
# To illustrate the use of this filter, we consider a reaching task. We
# assume two underlying system states s=1="Not Moving"=NM and s=2="Moving"=M.
# Under the "Not Moving" the position of the arm remain constant,
# whereas in the "Moving" state, the position and velocities evolved based
# on the arm acceleration that is modeled as a gaussian white noise
# process.
#
# Under both the "Moving" and "Not Moving" states, the arm evolution state
# vector is

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 39

_ = section_index


In [ ]:
# MATLAB section 40/43 for nSTATPaperExamples: Section

# %
#
# $${\bf{x}} = {[x,y,{v_x},{v_y},{a_x},{a_y}]^T}$$
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 40

_ = section_index


In [ ]:
# MATLAB section 41/43 for nSTATPaperExamples: Generated Simulated Arm Reach

# % Generated Simulated Arm Reach
#
# MATLAB: clear all;
# MATLAB: [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:     getPaperDataDirs();
# MATLAB: close all;
# MATLAB: delta=0.001;
# MATLAB: Tmax=2;
# MATLAB: time=0:delta:Tmax;
# MATLAB: A{2} = [1 0 delta 0     delta^2/2   0;
# MATLAB:         0 1 0     delta 0           delta^2/2;
# MATLAB:         0 0 1     0     delta       0;
# MATLAB:         0 0 0     1     0           delta;
# MATLAB:         0 0 0     0     1           0;
# MATLAB:         0 0 0     0     0           1];
#
# MATLAB: A{1} = [1 0 0 0 0 0;
# MATLAB:         0 1 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0];
# MATLAB: A{1} = [1 0;
# MATLAB:         0 1];
#
# MATLAB: Px0{2} =1e-6*eye(6,6);
# MATLAB: Px0{1} =1e-6*eye(2,2);
#
# MATLAB: minCovVal = 1e-12;
# MATLAB: covVal = 1e-3;
#
#
#
# MATLAB: Q{2}=[minCovVal     0   0     0   0       0;
# MATLAB:       0       minCovVal 0     0   0       0;
# MATLAB:       0       0   minCovVal   0   0       0;
# MATLAB:       0       0   0     minCovVal 0       0;
# MATLAB:       0       0   0     0   covVal      0;
# MATLAB:       0       0   0     0   0       covVal];
#
# MATLAB: Q{1}=minCovVal*eye(2,2);
#
# MATLAB: mstate = zeros(1,length(time));
# MATLAB: ind{1}=1:2;
# MATLAB: ind{2}=1:6;
#
# Acceleration model
# MATLAB: X=zeros(max([size(A{1},1),size(A{2},1)]),length(time));
# MATLAB: p_ij = [.998 .002;
# MATLAB:         .001 .999];
#
# MATLAB: for i = 1:length(time)
#
# MATLAB:     if(i==1)
# MATLAB:         mstate(i) = 1;
# MATLAB:     else
# MATLAB:        if(rand(1,1)<p_ij(mstate(i-1),mstate(i-1)))
# MATLAB:             mstate(i) = mstate(i-1);
# MATLAB:        else
# MATLAB:            if(mstate(i-1)==1)
# MATLAB:                mstate(i) = 2;
# MATLAB:            else
# MATLAB:                mstate(i) = 1;
# MATLAB:            end
# MATLAB:        end
# MATLAB:     end
# MATLAB:     st = mstate(i);
# MATLAB:     R=chol(Q{st});
# MATLAB:     if(i<length(time))
# MATLAB:         X(ind{st},i+1) = A{st}*X(ind{st},i) + R*randn(length(ind{st}),1);
# MATLAB:     end
#
# MATLAB: end

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 41

_ = section_index


In [ ]:
# MATLAB section 42/43 for nSTATPaperExamples: Section

# %
# save paperHybridFilterExample time Tmax delta mstate X p_ij ind A Q Px0
# MATLAB: load(fullfile(fileparts(which('nSTATPaperExamples')),'paperHybridFilterExample.mat'));
# MATLAB: Q{1}=minCovVal*eye(2,2);
# MATLAB: numCells=40;
# MATLAB: close all;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:     scrsz(3)*.8 scrsz(4)*.9]);
# MATLAB: subplot(4,2,[1 3]);
# MATLAB: plot(100*X(1,:),100*X(2,:),'k','Linewidth',2); hx=xlabel('X [cm]');
# MATLAB: hy=ylabel('Y [cm]');  hold on;
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title('Reach Path','FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB: hold on;
# MATLAB: h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',16);
# MATLAB: h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',16);
# MATLAB: legend([h1 h2],'Start','Finish','Location','NorthEast');
#
#
#
# MATLAB: subplot(4,2,[6 8]);
# MATLAB: plot(time,mstate,'k','Linewidth',2); axis tight; v=axis;
# MATLAB: axis([v(1) v(2) 0 3]);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('state');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: set(gca,'YTick',[1 2],'YTickLabel',{'N','M'})
# MATLAB: title('Discrete Movement State','FontWeight','bold','Fontsize',14,...
# MATLAB:     'FontName','Arial');
#
# MATLAB: subplot(4,2,5);
# MATLAB: h1=plot(time,100*X(1,1:end),'k','Linewidth',2); hold on;
# MATLAB: h2=plot(time,100*X(2,1:end),'k-.','Linewidth',2);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('Position [cm]');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: h_legend=legend([h1,h2],'x','y','Location','NorthEast');
# MATLAB: set(h_legend,'FontSize',14)
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
#
#
# MATLAB: subplot(4,2,7);
# MATLAB: h1=plot(time,100*X(3,1:end),'k','Linewidth',2); hold on;
# MATLAB: h2=plot(time,100*X(4,1:end),'k-.','Linewidth',2);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('Velocity [cm/s]');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: h_legend=legend([h1,h2],'v_{x}','v_{y}','Location','NorthEast');
# MATLAB: set(h_legend,'FontSize',14)
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
#
# MATLAB: meanMu = log(10*delta);  % baseline firing rate
# MATLAB: MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
# MATLAB: coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...
# MATLAB:     0*randn(numCells,2)];
# Add realization by thinning with history
# MATLAB: dataMat = [ones(size(X,2),1),X(:,1:end)'];
# Generate M1 cells
# MATLAB: clear lambda tempSpikeColl lambdaCIF n;
# MATLAB: fitType ='binomial';
# matlabpool open;
# MATLAB:  for i=1:numCells
# MATLAB:      tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:      if(strcmp(fitType,'binomial'));
# MATLAB:         lambdaData = tempData./(1+tempData);
# MATLAB:      else
# MATLAB:         lambdaData = tempData;
# MATLAB:      end
# MATLAB:      lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:          '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:          {strcat('\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:      maxTimeRes = 0.001;
# MATLAB:      tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);
# MATLAB:      n{i} = tempSpikeColl{i}.getNST(1);
# MATLAB:      n{i}.setName(num2str(i));
# MATLAB:  end
# MATLAB:  spikeColl = nstColl(n);
# MATLAB:  subplot(4,2,[2 4]);
# MATLAB: spikeColl.plot;
# MATLAB: set(gca,'xtick',[],'xtickLabel',[],'ytickLabel',[]);
# MATLAB: title('Neural Raster','FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB: hx=xlabel('time [s]','Interpreter','none');
# MATLAB: hy=ylabel('Cell Number','Interpreter','none');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# close all;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 42

_ = section_index


In [ ]:
# MATLAB section 43/43 for nSTATPaperExamples: Simulate Neural Firing

# % Simulate Neural Firing
# We simulate a population of neurons that fire in response to the movement
# velocity (x and y coorinates)
#
# Use the data to estimate the process noise for the moving case and
# non-moving case
#
# MATLAB: nonMovingInd = intersect(find(X(5,:)==0),find(X(6,:)==0));
# MATLAB: movingInd=setdiff(1:size(X,2),nonMovingInd);
# MATLAB: Q{2}=diag(var(diff(X(:,movingInd),[],2),[],2));
# MATLAB: Q{2}(1:4,1:4)=0;
# MATLAB: varNV=diag(var(diff(X(:,nonMovingInd),[],2),[],2));
# MATLAB: Q{1} = varNV(1:2,1:2);
# MATLAB: close all; clear S_est X_est MU_est S_estNT X_estNT MU_estNT;
# MATLAB: numExamples = 20;
# MATLAB: numCells=40;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:     scrsz(3)*.9 scrsz(4)*.9]);
#
# MATLAB: for n=1:numExamples
# MATLAB:     meanMu = log(10*delta);  % baseline firing rate
# MATLAB:     MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
# MATLAB:     coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...
# MATLAB:         0*randn(numCells,2)];
#
#
#
# Add realization by thinning with history
# MATLAB:     dataMat = [ones(size(X,2),1),X(:,1:end)'];
# Generate M1 cells
# MATLAB:     clear lambda tempSpikeColl lambdaCIF nst;
# MATLAB:     fitType ='binomial';
# matlabpool open;
# MATLAB:      for i=1:numCells
# MATLAB:          tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:          if(strcmp(fitType,'binomial'))
# MATLAB:             lambdaData = tempData./(1+tempData);
# MATLAB:          else
# MATLAB:             lambdaData = tempData;
# MATLAB:          end
# MATLAB:          lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:              '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:              {strcat('\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:          maxTimeRes = 0.001;
# MATLAB:          tempSpikeColl{i} = ...
# MATLAB:              CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);
# MATLAB:          nst{i} = tempSpikeColl{i}.getNST(1);
# MATLAB:          nst{i}.setName(num2str(i));
# MATLAB:      end
#
# Decode the x-y trajectory
#
# Enforce that the maximum time resolution is delta
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     spikeColl.resample(1/delta);
# MATLAB:     dN = spikeColl.dataToMatrix;
# MATLAB:     dN(dN>1)=1; %Avoid more than 1 spike per bin.
#
# Starting states are equally probable
# MATLAB:     Mu0=.5*ones(size(p_ij,1),1);
# MATLAB:     clear x0 yT clear Pi0 PiT;
# MATLAB:     x0{1} = X(ind{1},1);
# MATLAB:     yT{1} = X(ind{1},end);
# MATLAB:     Pi0    = Px0;
# MATLAB:     PiT{1} = 1e-9*eye(size(x0{1},1),size(x0{1},1));
#
# MATLAB:     x0{2} = X(ind{2},1);
# MATLAB:     yT{2} = X(ind{2},end);
# MATLAB:     PiT{2} = 1e-9*eye(size(x0{2},1),size(x0{2},1));
#
#
# Run the Hybrid Point Process Filter
# MATLAB:     [S_est, X_est, W_est, MU_est, X_s, W_s,pNGivenS]=...
# MATLAB:         DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...
# MATLAB:         coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0,Pi0, yT,PiT);
# MATLAB:     [S_estNT, X_estNT, W_estNT, MU_estNT, X_sNT, W_sNT,pNGivenSNT]=...
# MATLAB:         DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...
# MATLAB:         coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0,Pi0);
#
# Store the results for computing relevant statistics later
# MATLAB:     X_estAll(:,:,n) = X_est;
# MATLAB:     X_estNTAll(:,:,n) = X_estNT;
# MATLAB:     S_estAll(n,:)=S_est;
# MATLAB:     S_estNTAll(n,:)=S_estNT;
# MATLAB:     MU_estAll(:,:,n)=MU_est;
# MATLAB:     MU_estNTAll(:,:,n) = MU_estNT;
#
#
# State Estimate
# MATLAB:     subplot(4,3,[1 4]);
# MATLAB:     plot(time,mstate,'k','LineWidth',3); hold all;
# MATLAB:     plot(time,S_est,'b-.','Linewidth',.5);
# MATLAB:     plot(time,S_estNT,'g-.','Linewidth',.5); axis tight; v=axis;
# MATLAB:     axis([v(1) v(2) 0.5 2.5]);
#
# Movement State Probability (Non-movement State probability is 1-Pr(Movement))
# MATLAB:     subplot(4,3,[7 10]);
# MATLAB:     plot(time,MU_est(2,:),'b-.','Linewidth',.5);  hold on;
# MATLAB:     plot(time,MU_estNT(2,:),'g-.','Linewidth',.5);  hold on;
# MATLAB:     axis([min(time) max(time) 0 1.1]);
#
# The movement path
# MATLAB:     subplot(4,3,[2 3 5 6]);
# MATLAB:     h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;
# MATLAB:     h2=plot(100*X_est(1,:)',100*X_est(2,:)','b-.'); hold all;
# MATLAB:     h3=plot(100*X_estNT(1,:)',100*X_estNT(2,:)','g-.');
#
# X-Position
# MATLAB:     subplot(4,3,8);
# MATLAB:     h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(1,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(1,:)','g-.');
#
# Y-Position
# MATLAB:     subplot(4,3,9);
# MATLAB:     h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(2,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(2,:)','g-.');
#
# X-Velocity
# MATLAB:     subplot(4,3,11);
# MATLAB:     h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(3,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(3,:)','g-.');
#
# MATLAB:     subplot(4,3,12);
# MATLAB:     h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(4,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(4,:)','g-.');
#
#
#
#
# MATLAB: end
#
#
# Save all the example Data
# save Experiment6ReachExamples X_estAll X_estNTAll S_estAll ...
# S_estNTAll MU_estAll MU_estNTAll;
#
# load Experiment6ReachExamples;
#
# Mean Discrete State Estimate
# MATLAB:     subplot(4,3,[1 4]);
# MATLAB:     hold all;
# MATLAB:     plot(time,mstate,'k','LineWidth',3);
# MATLAB:     plot(time,mean(S_estAll),'b','LineWidth',3);
# MATLAB:     plot(time,mean(S_estNTAll),'g','LineWidth',3);
# MATLAB:     set(gca,'xtick',[],'YTick',[1 2.1],'YTickLabel',{'N','M'});
# MATLAB:     hy=ylabel('state'); hx=xlabel('time [s]');
# MATLAB:     set([hy hx],'FontName', 'Arial','FontSize',10,'FontWeight','bold',...
# MATLAB:         'Interpreter','none');
# MATLAB:     title('Estimated vs. Actual State','FontWeight','bold','Fontsize',...
# MATLAB:         12,'FontName','Arial');
#
#
#
#
# Mean State Movement State Probability
# MATLAB:     subplot(4,3,[7 10]);
# MATLAB:     plot(time, mean(squeeze(MU_estAll(2,:,:)),2),'b','LineWidth',3);
# MATLAB:     hold on;
# MATLAB:     plot(time,mean(squeeze(MU_estNTAll(2,:,:)),2),'g','LineWidth',3);
# MATLAB:     hold on;
# MATLAB:     axis([min(time) max(time) 0 1.1]);
# MATLAB:     hx=xlabel('time [s]'); hy=ylabel('P(s(t)=M | data)');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Probability of State','FontWeight','bold','Fontsize',12,...
# MATLAB:         'FontName','Arial');
#
# Mean movement path
# MATLAB:     subplot(4,3,[2 3 5 6]);
# MATLAB:     h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;
# MATLAB:     mXestAll=mean(100*X_estAll,3);
# MATLAB:     mXestNTAll=mean(100*X_estNTAll,3);
# MATLAB:     plot(mXestAll(1,:),mXestAll(2,:),'b','Linewidth',3);
# MATLAB:     plot(mXestNTAll(1,:),mXestNTAll(2,:),'g','Linewidth',3);
# MATLAB:     hx=xlabel('x [cm]'); hy=ylabel('y [cm]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
#
# MATLAB:     h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',14); hold on;
# MATLAB:     h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',14);
# MATLAB:     legend([h1 h2],'Start','Finish','Location','NorthEast');
# MATLAB:     title('Estimated vs. Actual Reach Path','FontWeight','bold',...
# MATLAB:         'Fontsize',12,'FontName','Arial');
#
#
# Mean X-Positon
# MATLAB:     subplot(4,3,8);
# MATLAB:     h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(1,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(1,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('x(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# Mean Y-Position
# MATLAB:     subplot(4,3,9);
# MATLAB:     h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(2,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(2,:),'g','LineWidth',3); hold on;
# MATLAB:     h_legend=legend([h1(1) h2(1) h3(1)],'Actual','PPAF+Goal',...
# MATLAB:         'PPAF','Location','SouthEast');
# MATLAB:     hy=ylabel('y(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
# MATLAB:     set(h_legend,'FontSize',10)
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)-.40 pos(2)+.51 pos(3:4)]);
#
# Mean X-Velocity
# MATLAB:     subplot(4,3,11);
# MATLAB:     h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(3,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(3,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('v_{x}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# Mean Y-Velocity
# MATLAB:     subplot(4,3,12);
# MATLAB:     h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(4,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(4,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('v_{y}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# Parity contract scalars for MATLAB/Python verification.
# MATLAB: parity = struct();
# MATLAB: if exist('numCells','var')
# MATLAB:     parity.num_cells = numCells;
# MATLAB: end
# MATLAB: if exist('numRealizations','var')
# MATLAB:     parity.num_realizations = numRealizations;
# MATLAB: end
#
# MATLAB: function [dataDir,mEPSCDir,explicitStimulusDir,psthDir,placeCellDataDir] = ...
# MATLAB:     getPaperDataDirs()
# Resolve local data folders robustly when Live Editor executes from a temp
# location (e.g., /private/var/.../T).
# MATLAB: candidateRoots = {};
#
# MATLAB: scriptPath = mfilename('fullpath');
# MATLAB: if ~isempty(scriptPath)
# MATLAB:     candidateRoots = appendCandidateRoot(candidateRoots, fileparts(fileparts(scriptPath)));
# MATLAB: end
#
# MATLAB: paperPath = which('nSTATPaperExamples');
# MATLAB: if ~isempty(paperPath)
# MATLAB:     candidateRoots = appendCandidateRoot(candidateRoots, fileparts(fileparts(paperPath)));
# MATLAB: end
#
# MATLAB: installPath = which('nSTAT_Install');
# MATLAB: if ~isempty(installPath)
# MATLAB:     candidateRoots = appendCandidateRoot(candidateRoots, fileparts(installPath));
# MATLAB: end
#
# MATLAB: try
# MATLAB:     activeFile = matlab.desktop.editor.getActiveFilename;
# MATLAB: catch
# MATLAB:     activeFile = '';
# MATLAB: end
# MATLAB: if ~isempty(activeFile)
# MATLAB:     candidateRoots = appendCandidateRoot(candidateRoots, fileparts(fileparts(activeFile)));
# MATLAB: end
#
# MATLAB: candidateRoots = appendCandidateRoot(candidateRoots, pwd);
#
# MATLAB: nSTATDir = '';
# MATLAB: for iRoot = 1:numel(candidateRoots)
# MATLAB:     candidateDataDir = fullfile(candidateRoots{iRoot}, 'data');
# MATLAB:     if exist(candidateDataDir, 'dir') == 7
# MATLAB:         nSTATDir = candidateRoots{iRoot};
# MATLAB:         break;
# MATLAB:     end
# MATLAB: end
#
# MATLAB: if isempty(nSTATDir)
# MATLAB:     error('nSTATPaperExamples:MissingInstallPath', ...
# MATLAB:         ['Could not resolve the nSTAT root path. Checked roots derived from ', ...
# MATLAB:          'mfilename, which(''nSTATPaperExamples''), which(''nSTAT_Install''), ', ...
# MATLAB:          'the active editor file, and pwd.']);
# MATLAB: end
#
# MATLAB: dataDir = fullfile(nSTATDir,'data');
# MATLAB: mEPSCDir = fullfile(dataDir,'mEPSCs');
# MATLAB: explicitStimulusDir = fullfile(dataDir,'Explicit Stimulus');
# MATLAB: psthDir = fullfile(dataDir,'PSTH');
# MATLAB: placeCellDataDir = fullfile(dataDir,'Place Cells');
#
# MATLAB: if exist(dataDir,'dir') ~= 7
# MATLAB:     error('nSTATPaperExamples:MissingDataDir', ...
# MATLAB:         'Could not find local nSTAT data folder at %s', dataDir);
# MATLAB: end
# MATLAB: end
#
# MATLAB: function roots = appendCandidateRoot(roots, startDir)
# MATLAB: if isempty(startDir)
# MATLAB:     return;
# MATLAB: end
#
# MATLAB: thisDir = startDir;
# MATLAB: while true
# MATLAB:     if ~any(strcmp(roots, thisDir))
# MATLAB:         roots{end+1} = thisDir; %#ok<AGROW>
# MATLAB:     end
# MATLAB:     parentDir = fileparts(thisDir);
# MATLAB:     if strcmp(parentDir, thisDir)
# MATLAB:         break;
# MATLAB:     end
# MATLAB:     thisDir = parentDir;
# MATLAB: end
# MATLAB: end
#

# Python translation execution block for this helpfile.

# nSTATPaperExamples: multi-section paper-style workflow summary.
import json
from pathlib import Path
from scipy.io import loadmat
from nstat.compat.matlab import Analysis, DecodingAlgorithms, nspikeTrain, nstColl


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
fixture_root = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold"
shared_root = repo_root / "data" / "shared" / "matlab_gold_20260302"
mEPSCDir = shared_root / "mEPSCs"

# -------------------------------------------------------------------------
# Experiment 1: mEPSCs - Constant Magnesium Concentration.
# MATLAB reference:
#   - epsc2.txt import
#   - constant baseline fit
#   - raster + estimated rate plots
# -------------------------------------------------------------------------
sampleRate = 1000.0
delta = 1.0 / sampleRate

epsc2 = np.genfromtxt(mEPSCDir / "epsc2.txt", skip_header=1)
spikeTimes_const = np.asarray(epsc2[:, 1], dtype=float) / sampleRate
nstConst = nspikeTrain(spikeTimes_const)
spikeCollConst = nstColl([nstConst])

timeConst = np.arange(0.0, float(spikeTimes_const.max()) + delta, delta)
bin_edges_const = np.append(timeConst, timeConst[-1] + delta)
dN_const, _ = np.histogram(spikeTimes_const, bins=bin_edges_const)

X_const = np.ones((dN_const.size, 1), dtype=float)
fitConst = Analysis.fitGLM(X=X_const, y=dN_const.astype(float), fitType="poisson", dt=delta)
lambdaConst = np.asarray(fitConst.predict(X_const), dtype=float).reshape(-1) / delta
lambdaConstMean = float(np.mean(lambdaConst))

fig1, axes1 = plt.subplots(2, 2, figsize=(12.0, 8.2))
axes1[0, 0].eventplot([spikeTimes_const], colors="k", linelengths=0.9)
axes1[0, 0].set_title("Constant Mg: neural raster")
axes1[0, 0].set_xlabel("time [s]")
axes1[0, 0].set_ylabel("mEPSCs")

axes1[0, 1].plot(timeConst, lambdaConst, "b", linewidth=1.5, label="GLM constant-rate estimate")
axes1[0, 1].axhline(lambdaConstMean, color="r", linestyle="--", linewidth=1.0, label="mean rate")
axes1[0, 1].set_title("Constant Mg: estimated rate")
axes1[0, 1].set_xlabel("time [s]")
axes1[0, 1].set_ylabel("rate [spikes/sec]")
axes1[0, 1].legend(loc="upper right", fontsize=8)

isi_const = np.diff(spikeTimes_const)
axes1[1, 0].hist(isi_const, bins=60, color="0.35", alpha=0.85)
axes1[1, 0].set_title("Constant Mg: ISI histogram")
axes1[1, 0].set_xlabel("inter-spike interval [s]")
axes1[1, 0].set_ylabel("count")

axes1[1, 1].plot(np.arange(dN_const.size) * delta, dN_const, "k", linewidth=0.8)
axes1[1, 1].set_title("Constant Mg: binned spike train")
axes1[1, 1].set_xlabel("time [s]")
axes1[1, 1].set_ylabel("spike count / bin")
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# Experiment 1: mEPSCs - Varying Magnesium Concentration (piecewise model).
# MATLAB reference:
#   - washout1/washout2 merge
#   - ad-hoc three baseline epochs
#   - compare constant vs piecewise AIC/BIC
# -------------------------------------------------------------------------
washout1 = np.genfromtxt(mEPSCDir / "washout1.txt", skip_header=1)
washout2 = np.genfromtxt(mEPSCDir / "washout2.txt", skip_header=1)

spikeTimes1 = 260.0 + np.asarray(washout1[:, 1], dtype=float) / sampleRate
spikeTimes2 = np.sort(np.asarray(washout2[:, 1], dtype=float)) / sampleRate + 745.0
spikeTimes_var = np.concatenate([spikeTimes1, spikeTimes2])
nstVar = nspikeTrain(spikeTimes_var)
spikeCollVar = nstColl([nstVar])

timeVar = np.arange(260.0, float(spikeTimes_var.max()) + delta, delta)
bin_edges_var = np.append(timeVar, timeVar[-1] + delta)
dN_var, _ = np.histogram(spikeTimes_var, bins=bin_edges_var)

timeInd1 = int(np.searchsorted(timeVar, 495.0, side="right"))
timeInd2 = int(np.searchsorted(timeVar, 765.0, side="right"))

constantRate = np.ones(timeVar.size, dtype=float)
rate1 = np.zeros(timeVar.size, dtype=float)
rate2 = np.zeros(timeVar.size, dtype=float)
rate3 = np.zeros(timeVar.size, dtype=float)
rate1[:timeInd1] = 1.0
rate2[timeInd1:timeInd2] = 1.0
rate3[timeInd2:] = 1.0

X_var_const = constantRate.reshape(-1, 1)
X_var_piecewise = np.column_stack([rate1, rate2, rate3])
fitVarConst = Analysis.fitGLM(X=X_var_const, y=dN_var.astype(float), fitType="poisson", dt=delta)
fitVarPiecewise = Analysis.fitGLM(X=X_var_piecewise, y=dN_var.astype(float), fitType="poisson", dt=delta)
lambdaVarConst = np.asarray(fitVarConst.predict(X_var_const), dtype=float).reshape(-1) / delta
lambdaVarPiecewise = np.asarray(fitVarPiecewise.predict(X_var_piecewise), dtype=float).reshape(-1) / delta

dAIC_piecewise = float(fitVarConst.aic() - fitVarPiecewise.aic())
dBIC_piecewise = float(fitVarConst.bic() - fitVarPiecewise.bic())

fig2, axes2 = plt.subplots(2, 2, figsize=(12.2, 8.4))
axes2[0, 0].eventplot([spikeTimes_var], colors="k", linelengths=0.9)
axes2[0, 0].axvline(495.0, color="r", linewidth=1.5)
axes2[0, 0].axvline(765.0, color="r", linewidth=1.5)
axes2[0, 0].set_title("Varying Mg: neural raster + epoch boundaries")
axes2[0, 0].set_xlabel("time [s]")
axes2[0, 0].set_ylabel("mEPSCs")

axes2[0, 1].plot(timeVar, lambdaVarConst, "b", linewidth=1.1, label="constant baseline")
axes2[0, 1].plot(timeVar, lambdaVarPiecewise, "g", linewidth=1.1, label="piecewise baseline")
axes2[0, 1].set_title("Varying Mg: model rates")
axes2[0, 1].set_xlabel("time [s]")
axes2[0, 1].set_ylabel("rate [spikes/sec]")
axes2[0, 1].legend(loc="upper right", fontsize=8)

axes2[1, 0].plot(timeVar, dN_var, "0.25", linewidth=0.7)
axes2[1, 0].set_title("Varying Mg: binned spike train")
axes2[1, 0].set_xlabel("time [s]")
axes2[1, 0].set_ylabel("spike count / bin")

axes2[1, 1].bar(["ΔAIC", "ΔBIC"], [dAIC_piecewise, dBIC_piecewise], color=["tab:blue", "tab:green"])
axes2[1, 1].axhline(0.0, color="k", linewidth=0.8)
axes2[1, 1].set_title("Piecewise minus constant model quality")
axes2[1, 1].set_ylabel("improvement (>0 favors piecewise)")
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# Experiment 5 proxies: stimulus decoding + place-cell decoding + PSTH CI.
# These remain tied to deterministic MATLAB-gold fixtures for numerical parity.
# -------------------------------------------------------------------------
m_pp = loadmat(fixture_root / "PPSimExample_gold.mat")
X_pp = np.asarray(m_pp["X"], dtype=float)
y_pp = np.asarray(m_pp["y"], dtype=float).reshape(-1)
dt_pp = float(np.asarray(m_pp["dt"], dtype=float).reshape(-1)[0])
b_pp = np.asarray(m_pp["b"], dtype=float).reshape(-1)
expected_rate_pp = np.asarray(m_pp["expected_rate"], dtype=float).reshape(-1)

fit_pp = Analysis.fitGLM(X=X_pp, y=y_pp, fitType="poisson", dt=dt_pp)
rate_hat_pp = np.asarray(fit_pp.predict(X_pp), dtype=float).reshape(-1)
coef_pp = np.concatenate([[float(fit_pp.intercept)], np.asarray(fit_pp.coefficients, dtype=float)])
coef_err_pp = float(np.linalg.norm(coef_pp - b_pp))
rate_rel_err_pp = float(
    np.mean(np.abs(rate_hat_pp - expected_rate_pp) / np.maximum(np.abs(expected_rate_pp), 1e-12))
)

m_dec = loadmat(fixture_root / "DecodingExampleWithHist_gold.mat")
spike_counts = np.asarray(m_dec["spike_counts"], dtype=float)
tuning = np.asarray(m_dec["tuning"], dtype=float)
transition = np.asarray(m_dec["transition"], dtype=float)
expected_decoded = np.asarray(m_dec["expected_decoded"], dtype=int).reshape(-1)
expected_post = np.asarray(m_dec["expected_posterior"], dtype=float)

decoded_hist, posterior_hist = DecodingAlgorithms.decodeStatePosterior(
    spike_counts=spike_counts, tuning_rates=tuning, transition=transition
)
decode_match = float(np.mean(decoded_hist == expected_decoded))
posterior_max_abs = float(np.max(np.abs(posterior_hist - expected_post)))

m_pc = loadmat(fixture_root / "HippocampalPlaceCellExample_gold.mat")
spike_counts_pc = np.asarray(m_pc["spike_counts_pc"], dtype=float)
tuning_curves = np.asarray(m_pc["tuning_curves"], dtype=float)
expected_weighted = np.asarray(m_pc["expected_decoded_weighted"], dtype=float).reshape(-1)

decoded_weighted = DecodingAlgorithms.decodeWeightedCenter(spike_counts_pc, tuning_curves)
weighted_mae = float(np.mean(np.abs(decoded_weighted - expected_weighted)))
weighted_max_err = float(np.max(np.abs(decoded_weighted - expected_weighted)))

m_psth = loadmat(fixture_root / "PSTHEstimation_gold.mat")
spike_matrix_psth = np.asarray(m_psth["spike_matrix_psth"], dtype=float)
alpha_psth = float(np.asarray(m_psth["alpha_psth"], dtype=float).reshape(-1)[0])
expected_rate_psth = np.asarray(m_psth["expected_rate_psth"], dtype=float).reshape(-1)
expected_prob_psth = np.asarray(m_psth["expected_prob_psth"], dtype=float)
expected_sig_psth = np.asarray(m_psth["expected_sig_psth"], dtype=int)

rate_psth, prob_psth, sig_psth = DecodingAlgorithms.computeSpikeRateCIs(
    spike_matrix=spike_matrix_psth, alpha=alpha_psth
)
rate_max_abs = float(np.max(np.abs(rate_psth - expected_rate_psth)))
prob_max_abs = float(np.max(np.abs(prob_psth - expected_prob_psth)))
sig_mismatch = int(np.sum(np.abs(sig_psth - expected_sig_psth)))

audit_path = fixture_root / "nSTATPaperExamples_audit_gold.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit_alignment = str(audit.get("alignment_status", ""))
audit_code_lines = int(audit.get("matlab_code_lines", 0))
audit_ref_images = int(audit.get("matlab_reference_image_count", 0))

fig3, axes3 = plt.subplots(2, 3, figsize=(13.2, 8.6))
axes3[0, 0].plot(expected_rate_pp[:1200], "k", linewidth=1.0, label="MATLAB gold")
axes3[0, 0].plot(rate_hat_pp[:1200], "tab:blue", linewidth=1.0, label="Python fit")
axes3[0, 0].set_title("Stimulus proxy: GLM rate fit")
axes3[0, 0].legend(loc="upper right", fontsize=8)

axes3[0, 1].plot(expected_decoded[:180], "k", linewidth=1.0, label="MATLAB decoded")
axes3[0, 1].plot(decoded_hist[:180], "tab:green", linewidth=0.9, label="Python decoded")
axes3[0, 1].set_title("Decode-with-history path")
axes3[0, 1].legend(loc="upper right", fontsize=8)

im0 = axes3[0, 2].imshow(np.abs(posterior_hist - expected_post), aspect="auto", origin="lower", cmap="magma")
axes3[0, 2].set_title("Posterior absolute error")
fig3.colorbar(im0, ax=axes3[0, 2], fraction=0.045, pad=0.02)

axes3[1, 0].plot(expected_weighted, "k", linewidth=1.0, label="MATLAB weighted")
axes3[1, 0].plot(decoded_weighted, "tab:red", linewidth=0.9, label="Python weighted")
axes3[1, 0].set_title("Place-cell weighted decode")
axes3[1, 0].legend(loc="upper right", fontsize=8)

field = tuning_curves[6].reshape(5, 8)
im1 = axes3[1, 1].imshow(field, origin="lower", cmap="jet", aspect="auto")
axes3[1, 1].set_title("Example place field (unit 7)")
fig3.colorbar(im1, ax=axes3[1, 1], fraction=0.045, pad=0.02)

im2 = axes3[1, 2].imshow(prob_psth, origin="lower", cmap="gray_r", aspect="auto")
yy, xx = np.where(sig_psth > 0)
if xx.size:
    axes3[1, 2].plot(xx, yy, "r*", markersize=3)
axes3[1, 2].set_title("Trial significance matrix")
fig3.colorbar(im2, ax=axes3[1, 2], fraction=0.045, pad=0.02)
plt.tight_layout()
plt.show()

assert lambdaConstMean > 0.0
assert dAIC_piecewise >= 0.0
assert dBIC_piecewise >= 0.0
assert coef_err_pp < 0.7
assert rate_rel_err_pp < 0.30
assert decode_match >= 1.0
assert posterior_max_abs < 1e-9
assert weighted_mae < 1e-10
assert weighted_max_err < 1e-10
assert rate_max_abs < 1e-10
assert prob_max_abs < 1e-10
assert sig_mismatch == 0
assert audit_alignment == "validated"
assert audit_code_lines > 1000

CHECKPOINT_METRICS = {
    "const_mean_rate": float(lambdaConstMean),
    "dAIC_piecewise": float(dAIC_piecewise),
    "dBIC_piecewise": float(dBIC_piecewise),
    "coef_error_pp": float(coef_err_pp),
    "rate_rel_err_pp": float(rate_rel_err_pp),
    "decode_match": float(decode_match),
    "weighted_mae": float(weighted_mae),
    "psth_rate_max_abs": float(rate_max_abs),
    "sig_mismatch": float(sig_mismatch),
    "matlab_code_lines": float(audit_code_lines),
    "matlab_ref_images": float(audit_ref_images),
}
CHECKPOINT_LIMITS = {
    "const_mean_rate": (0.01, 20000.0),
    "dAIC_piecewise": (0.0, 5.0e4),
    "dBIC_piecewise": (0.0, 5.0e4),
    "coef_error_pp": (0.0, 0.7),
    "rate_rel_err_pp": (0.0, 0.30),
    "decode_match": (1.0, 1.0),
    "weighted_mae": (0.0, 1e-10),
    "psth_rate_max_abs": (0.0, 1e-10),
    "sig_mismatch": (0.0, 0.0),
    "matlab_code_lines": (1000.0, 5000.0),
    "matlab_ref_images": (1.0, 1000.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
